# 🎯 Senior Analytics Interview
## In-App Event Correlation & Paid Targeting Analysis

---

**Difficulty:** Senior Data Scientist / Staff Analytics Engineer  
**Duration:** 90 – 120 minutes  
**Format:** Live coding + business interpretation  
**Interviewer:** Senior Product Analytics Lead  

---

> *"This interview simulates production analytics at a consumer app company with 10M+ MAU, multi-channel paid acquisition, and a complex attribution environment. Expect ambiguity — that is intentional."*


## Section 1 — Business Scenario

### App: **ShopFlash** — Social Commerce Mobile App

**What it is:**  
ShopFlash is a mobile-first social commerce app (iOS + Android) where users discover, share, and purchase products via short video feeds, live streams, and curated collections. Think TikTok Shop meets Pinterest.

**Monetization model:**
- In-app purchases (physical & digital products)
- Premium subscription (ShopFlash Pro — ad-free + early access)
- Affiliate commissions from brand partnerships
- In-app advertising (brands paying for product placements)

**Scale:**
- ~10M MAU, 2.5M DAU
- 15 active paid campaigns across Facebook, Google, TikTok, Apple Search Ads
- $400K/month UA budget
- Attribution via AppsFlyer (7-day click, 1-day view-through)

**Analytics Challenges:**
- Multi-touch attribution across 4+ channels
- Bot traffic contaminating campaign data (~5% installs)
- Delayed events from offline purchases (up to 48h lag)
- Timezone mismatches between server logs and client logs
- Users reinstalling with different device IDs
- Campaign cannibalization between paid channels
- iOS 14.5+ SKAN attribution degrading signal quality

**Your role today:** Staff Analytics Engineer running this interview as a take-home + live coding hybrid.


---
## Section 2 — Dataset Generation

All datasets are generated synthetically but designed to mirror production-scale mobile app analytics data. Each table contains intentional data quality issues you will encounter in real systems.

**Tables:**
1. `users` — user profiles with attribution
2. `app_events` — raw in-app event stream
3. `campaigns` — campaign metadata
4. `ad_impressions` — impression log
5. `ad_clicks` — click log
6. `installs` — attributed installs
7. `purchases` — transaction log
8. `subscriptions` — subscription events
9. `audience_segments` — targeting segment membership


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency, ttest_ind, norm, mannwhitneyu
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from collections import Counter, defaultdict
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_palette('husl')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
pd.set_option('display.width', 200)

print('Libraries loaded successfully.')
print(f'pandas={pd.__version__}  numpy={np.__version__}')


In [ ]:
# ── Global constants ──────────────────────────────────────────────────────────
N_USERS          = 10_000
N_EVENTS         = 180_000
N_CAMPAIGNS      = 15
START_DATE       = pd.Timestamp('2024-01-01', tz='UTC')
END_DATE         = pd.Timestamp('2024-06-30', tz='UTC')
ATTR_WINDOW_DAYS = 7         # Last-click attribution window
SESSION_TIMEOUT  = 30        # minutes
BOT_RATE         = 0.05      # 5% bot installs
DUP_EVENT_RATE   = 0.03      # 3% duplicate events injected

CAMPAIGN_IDS   = [f'CMP_{i:03d}' for i in range(1, N_CAMPAIGNS + 1)]
COUNTRIES      = ['US','IN','BR','UK','DE','FR','CA','AU','MX','JP']
COUNTRY_P      = [.35,.20,.10,.08,.06,.05,.05,.04,.04,.03]
PLATFORMS      = ['iOS','Android']
SOURCES        = ['facebook','google','tiktok','organic','referral','email','apple']
EVENT_NAMES    = [
    'app_open','signup','search','product_view','add_to_cart',
    'checkout_start','purchase','ad_click','video_watch','level_complete',
    'subscription_start','push_open','invite_sent','wishlist_add',
    'share','rating_given','cart_abandon','checkout_fail','refund','app_close'
]

print(f'Constants set. Simulating {N_USERS:,} users | {N_EVENTS:,} events | {N_CAMPAIGNS} campaigns')


In [ ]:
# ── Table 1: campaigns ────────────────────────────────────────────────────────
campaigns = pd.DataFrame({
    'campaign_id': CAMPAIGN_IDS,
    'campaign_name': [
        'Brand_Awareness_Q1','Performance_UA_Jan','Retargeting_Cart',
        'Lookalike_LTV','Seasonal_Sale_Feb','Influencer_Feb',
        'Google_UAC_Mar','Meta_Advantage_Mar','TikTok_GenZ_Apr',
        'Apple_Search_Apr','Email_Reactivation','Push_Winback_Mar',
        'Referral_Program','YouTube_Brand_May','ASA_Competitor_May'
    ],
    'source': [
        'facebook','google','facebook','facebook','email','influencer',
        'google','facebook','tiktok','apple','email','push',
        'referral','youtube','apple'
    ],
    'medium': [
        'cpc','cpc','cpc','cpc','email','social',
        'uac','advantage_plus','paid_social','search','email','push',
        'referral','video','search'
    ],
    'budget_usd': [
        50000,80000,30000,60000,40000,25000,
        90000,70000,45000,35000,5000,1000,
        10000,20000,28000
    ],
    'cpm_usd': [
        12.5,8.0,15.0,13.0,6.0,20.0,
        7.0,11.0,9.5,25.0,2.0,0.5,
        0.0,10.0,24.0
    ],
    'quality_score': [
        0.60,0.75,0.88,0.87,0.70,0.45,
        0.80,0.82,0.65,0.90,0.70,0.60,
        0.88,0.55,0.85
    ],
    'start_date': pd.to_datetime([
        '2024-01-01','2024-01-15','2024-01-01','2024-02-01','2024-02-14','2024-02-01',
        '2024-03-01','2024-03-01','2024-04-01','2024-04-01','2024-02-15','2024-03-01',
        '2024-01-01','2024-05-01','2024-05-01'
    ]),
    'end_date': pd.to_datetime([
        '2024-03-31','2024-06-30','2024-06-30','2024-06-30','2024-02-28','2024-03-31',
        '2024-06-30','2024-06-30','2024-06-30','2024-06-30','2024-04-30','2024-06-30',
        '2024-06-30','2024-06-30','2024-06-30'
    ])
})

print('campaigns shape:', campaigns.shape)
campaigns[['campaign_id','campaign_name','source','budget_usd','quality_score']].head(8)


In [ ]:
# ── Table 2: users ────────────────────────────────────────────────────────────
rng = np.random.default_rng(42)

install_dates_raw = pd.to_datetime(
    rng.integers(
        pd.Timestamp('2024-01-01').value,
        pd.Timestamp('2024-06-30').value,
        size=N_USERS
    )
).tz_localize('UTC')

attr_campaign = rng.choice(
    CAMPAIGN_IDS + [None],
    size=N_USERS,
    p=[0.063]*N_CAMPAIGNS + [0.055]
)

users = pd.DataFrame({
    'user_id':            [f'USR_{i:06d}' for i in range(1, N_USERS + 1)],
    'device_id':          [f'DEV_{rng.integers(1, int(N_USERS*1.3)):08d}' for _ in range(N_USERS)],
    'country':            rng.choice(COUNTRIES, N_USERS, p=COUNTRY_P),
    'platform':           rng.choice(PLATFORMS, N_USERS, p=[0.45, 0.55]),
    'device_type':        rng.choice(['phone','tablet'], N_USERS, p=[0.87, 0.13]),
    'operating_system':   rng.choice(['iOS 17','iOS 16','Android 14','Android 13','Android 12'], N_USERS, p=[0.28,0.17,0.25,0.20,0.10]),
    'app_version':        rng.choice(['3.2.0','3.3.0','3.4.0','4.0.0','4.1.0'], N_USERS, p=[0.05,0.10,0.25,0.35,0.25]),
    'install_date':       install_dates_raw,
    'acquisition_source': rng.choice(SOURCES, N_USERS, p=[0.25,0.20,0.15,0.22,0.08,0.05,0.05]),
    'attributed_campaign': attr_campaign,
    'is_bot':             rng.choice([True, False], N_USERS, p=[BOT_RATE, 1 - BOT_RATE]),
    'age_group':          rng.choice(['18-24','25-34','35-44','45-54','55+'], N_USERS, p=[0.25,0.35,0.20,0.12,0.08]),
    'gender':             rng.choice(['M','F','Other','Unknown'], N_USERS, p=[0.42,0.48,0.04,0.06]),
    'lifetime_value':     np.round(rng.exponential(28, N_USERS), 2),
})

# Inject real-world messiness: ~2% users have duplicate device_ids (reinstalls)
dup_mask = rng.random(N_USERS) < 0.02
users.loc[dup_mask, 'device_id'] = users.loc[dup_mask, 'device_id'].sample(frac=1).values

print('users shape:', users.shape)
print('Bot users:', users['is_bot'].sum())
print('Null attributed_campaign:', users['attributed_campaign'].isna().sum())
users.head(4)


In [ ]:
# ── Table 3: app_events ───────────────────────────────────────────────────────
# Realistic event probability distribution (not uniform)
event_probs = np.array([
    0.20, 0.05, 0.12, 0.15, 0.08,
    0.04, 0.03, 0.02, 0.07, 0.01,
    0.005,0.03, 0.01, 0.06,
    0.02, 0.01, 0.04, 0.02, 0.005, 0.04
])
event_probs = event_probs / event_probs.sum()

event_user_ids  = rng.choice(users['user_id'].values, N_EVENTS, replace=True)
event_names     = rng.choice(EVENT_NAMES, N_EVENTS, p=event_probs)

# Timestamps: cluster events within active hours (8am–11pm)
base_ts = pd.to_datetime(
    rng.integers(
        pd.Timestamp('2024-01-01', tz='UTC').value,
        pd.Timestamp('2024-06-30', tz='UTC').value,
        size=N_EVENTS
    )
)
hour_offsets = pd.to_timedelta(rng.integers(0, 15, N_EVENTS), unit='h')   # 0–15h offset simulates daytime clustering
event_timestamps = base_ts + hour_offsets

session_ids = [f'SESS_{rng.integers(1, N_EVENTS//4):08d}' for _ in range(N_EVENTS)]

app_events = pd.DataFrame({
    'event_id':        [f'EVT_{i:08d}' for i in range(1, N_EVENTS + 1)],
    'user_id':         event_user_ids,
    'session_id':      session_ids,
    'event_name':      event_names,
    'event_timestamp': event_timestamps,
    'platform':        rng.choice(PLATFORMS, N_EVENTS, p=[0.45, 0.55]),
    'app_version':     rng.choice(['3.2.0','3.3.0','3.4.0','4.0.0','4.1.0'], N_EVENTS, p=[0.05,0.10,0.25,0.35,0.25]),
    'country':         rng.choice(COUNTRIES, N_EVENTS, p=COUNTRY_P),
    'screen_name':     rng.choice(['home','feed','search','pdp','cart','checkout','profile','explore'], N_EVENTS),
    'revenue_usd':     np.where(
        event_names == 'purchase',
        np.round(rng.exponential(35, N_EVENTS), 2),
        np.nan
    ),
    'is_bot':          rng.choice([True, False], N_EVENTS, p=[BOT_RATE, 1 - BOT_RATE]),
})

# ── Inject data quality issues ────────────────────────────────────────────────
# 1. Duplicate events (3%)
dup_idx   = rng.choice(app_events.index, size=int(N_EVENTS * DUP_EVENT_RATE), replace=False)
dup_rows  = app_events.loc[dup_idx].copy()
dup_rows['event_timestamp'] += pd.to_timedelta(rng.integers(0, 30, len(dup_idx)), unit='s')   # slight jitter
app_events = pd.concat([app_events, dup_rows], ignore_index=True)

# 2. Missing timestamps (1%)
missing_ts_idx = rng.choice(app_events.index, size=int(len(app_events) * 0.01), replace=False)
app_events.loc[missing_ts_idx, 'event_timestamp'] = pd.NaT

# 3. Timezone inconsistency: some events stored in local time without tz info
no_tz_idx = rng.choice(app_events.index, size=int(len(app_events) * 0.05), replace=False)
app_events['event_timestamp'] = app_events['event_timestamp'].astype(object)
app_events.loc[no_tz_idx, 'event_timestamp'] = (
    app_events.loc[no_tz_idx, 'event_timestamp']
    .apply(lambda x: x.replace(tzinfo=None) if pd.notna(x) else x)
)

# 4. Corrupted/inconsistent event names
bad_name_idx = rng.choice(app_events.index, size=200, replace=False)
app_events.loc[bad_name_idx, 'event_name'] = rng.choice(['App_Open','PURCHASE','product view','Add To Cart'], 200)

print('app_events shape:', app_events.shape)
print('Null timestamps:', app_events['event_timestamp'].isna().sum())
print('Duplicate event_ids:', app_events['event_id'].duplicated().sum(), ' (intentional via dup rows)')
app_events.head(4)


In [ ]:
# ── Table 4: ad_impressions ───────────────────────────────────────────────────
N_IMPR = 400_000
impr_campaigns = rng.choice(CAMPAIGN_IDS, N_IMPR, p=np.array(campaigns['budget_usd']) / campaigns['budget_usd'].sum() + 1e-9)
impr_campaigns_norm = np.array(campaigns['budget_usd'], dtype=float)
impr_campaigns_norm[impr_campaigns_norm == 0] = 500
impr_campaigns_norm = impr_campaigns_norm / impr_campaigns_norm.sum()

impr_campaign_ids = rng.choice(CAMPAIGN_IDS, N_IMPR, p=impr_campaigns_norm)

ad_impressions = pd.DataFrame({
    'impression_id':   [f'IMP_{i:09d}' for i in range(1, N_IMPR + 1)],
    'campaign_id':     impr_campaign_ids,
    'user_id':         rng.choice(users['user_id'].values, N_IMPR, replace=True),
    'device_id':       [f'DEV_{rng.integers(1, int(N_USERS*1.3)):08d}' for _ in range(N_IMPR)],
    'impression_time': pd.to_datetime(
        rng.integers(pd.Timestamp('2024-01-01',tz='UTC').value, pd.Timestamp('2024-06-30',tz='UTC').value, N_IMPR)
    ),
    'platform':        rng.choice(PLATFORMS, N_IMPR, p=[0.45, 0.55]),
    'country':         rng.choice(COUNTRIES, N_IMPR, p=COUNTRY_P),
    'ad_format':       rng.choice(['video','carousel','static','story','interstitial'], N_IMPR, p=[0.35,0.25,0.20,0.12,0.08]),
    'cost_usd':        np.round(rng.exponential(0.015, N_IMPR), 4),
})

# ── Table 5: ad_clicks ────────────────────────────────────────────────────────
N_CLICKS = 28_000
click_imp_idx = rng.choice(ad_impressions.index, N_CLICKS, replace=False)
clic_rows = ad_impressions.loc[click_imp_idx, ['impression_id','campaign_id','user_id','device_id','platform','country']].copy()
clic_rows.columns = ['impression_id','campaign_id','user_id','device_id','platform','country']
clic_rows['click_id'] = [f'CLK_{i:08d}' for i in range(1, N_CLICKS + 1)]
clic_rows['click_time'] = (
    ad_impressions.loc[click_imp_idx, 'impression_time'].values
    + pd.to_timedelta(rng.integers(1, 120, N_CLICKS), unit='s')
)
clic_rows['cost_usd'] = np.round(rng.exponential(0.45, N_CLICKS), 4)

ad_clicks = clic_rows.reset_index(drop=True)

print('ad_impressions shape:', ad_impressions.shape)
print('ad_clicks shape:', ad_clicks.shape)
print(f'Overall CTR: {len(ad_clicks)/len(ad_impressions)*100:.2f}%')


In [ ]:
# ── Table 6: installs ─────────────────────────────────────────────────────────
N_INSTALLS = 8_500
install_click_idx = rng.choice(ad_clicks.index, int(N_INSTALLS * 0.72), replace=False)
inst_from_click   = ad_clicks.loc[install_click_idx, ['click_id','campaign_id','user_id','device_id','platform','country']].copy()
inst_from_click['install_source'] = 'paid'

# Organic installs
n_organic = N_INSTALLS - len(inst_from_click)
org_users = rng.choice(users['user_id'].values, n_organic, replace=False)
org_df = pd.DataFrame({
    'click_id':       [None] * n_organic,
    'campaign_id':    [None] * n_organic,
    'user_id':        org_users,
    'device_id':      [f'DEV_{rng.integers(1, int(N_USERS*1.3)):08d}' for _ in range(n_organic)],
    'platform':       rng.choice(PLATFORMS, n_organic, p=[0.45,0.55]),
    'country':        rng.choice(COUNTRIES, n_organic, p=COUNTRY_P),
    'install_source': ['organic'] * n_organic,
})

installs = pd.concat([inst_from_click, org_df], ignore_index=True)
installs['install_id']   = [f'INS_{i:07d}' for i in range(1, len(installs) + 1)]
installs['install_time'] = pd.to_datetime(
    rng.integers(pd.Timestamp('2024-01-01',tz='UTC').value, pd.Timestamp('2024-06-30',tz='UTC').value, len(installs))
)
# Bot installs
installs['is_bot_install'] = rng.choice([True,False], len(installs), p=[BOT_RATE, 1-BOT_RATE])
# Delayed attribution: 8% of paid installs attributed after click window
delayed_idx = rng.choice(inst_from_click.index, int(len(inst_from_click)*0.08), replace=False)
installs.loc[delayed_idx, 'campaign_id'] = None   # attribution broken by delay

print('installs shape:', installs.shape)
print('Paid:', (installs['install_source']=='paid').sum(), '| Organic:', (installs['install_source']=='organic').sum())
print('Bot installs:', installs['is_bot_install'].sum())

# ── Table 7: purchases ────────────────────────────────────────────────────────
purchase_users = rng.choice(users['user_id'].values, 12_000, replace=True)
purchases = pd.DataFrame({
    'purchase_id':    [f'PUR_{i:07d}' for i in range(1, 12_001)],
    'user_id':        purchase_users,
    'purchase_time':  pd.to_datetime(
        rng.integers(pd.Timestamp('2024-01-01',tz='UTC').value, pd.Timestamp('2024-06-30',tz='UTC').value, 12_000)
    ),
    'revenue_usd':    np.round(np.abs(rng.normal(45, 30, 12_000)), 2),
    'product_id':     [f'PROD_{rng.integers(1,500):04d}' for _ in range(12_000)],
    'product_category': rng.choice(['fashion','electronics','beauty','home','sports','food'], 12_000, p=[0.30,0.20,0.18,0.12,0.12,0.08]),
    'campaign_attributed': rng.choice(CAMPAIGN_IDS + [None], 12_000, p=[0.063]*N_CAMPAIGNS+[0.055]),
    'is_refunded':    rng.choice([True,False], 12_000, p=[0.04,0.96]),
})
# Inject missing revenue (broken tracking)
purchases.loc[rng.choice(purchases.index, 300, replace=False), 'revenue_usd'] = np.nan

print('purchases shape:', purchases.shape)
print('Missing revenue:', purchases['revenue_usd'].isna().sum())
print(f'Avg order value: ${purchases["revenue_usd"].mean():.2f}')


In [ ]:
# ── Table 8: subscriptions ────────────────────────────────────────────────────
sub_users = rng.choice(users['user_id'].values, 2_200, replace=False)
subscriptions = pd.DataFrame({
    'subscription_id': [f'SUB_{i:06d}' for i in range(1, 2_201)],
    'user_id':         sub_users,
    'plan_type':       rng.choice(['monthly','annual','trial'], 2_200, p=[0.50,0.30,0.20]),
    'plan_price_usd':  np.where(
        rng.choice(['monthly','annual','trial'], 2_200, p=[0.50,0.30,0.20]) == 'annual', 79.99,
        np.where(rng.random(2_200) < 0.5, 9.99, 0.0)
    ),
    'start_time': pd.to_datetime(
        rng.integers(pd.Timestamp('2024-01-01',tz='UTC').value, pd.Timestamp('2024-06-30',tz='UTC').value, 2_200)
    ),
    'status':          rng.choice(['active','cancelled','trial_expired'], 2_200, p=[0.58,0.28,0.14]),
    'campaign_attributed': rng.choice(CAMPAIGN_IDS + [None], 2_200, p=[0.063]*N_CAMPAIGNS+[0.055]),
})
subscriptions['end_time'] = subscriptions['start_time'] + pd.to_timedelta(
    np.where(subscriptions['plan_type']=='annual', 365,
    np.where(subscriptions['plan_type']=='monthly', 30, 14)), unit='D'
)

# ── Table 9: audience_segments ────────────────────────────────────────────────
all_uids = users['user_id'].values
segments = {
    'high_ltv_top10':    set(rng.choice(all_uids, 1000, replace=False)),
    'cart_abandoners':   set(rng.choice(all_uids, 2500, replace=False)),
    'lapsed_30d':        set(rng.choice(all_uids, 1800, replace=False)),
    'video_engagers':    set(rng.choice(all_uids, 3000, replace=False)),
    'new_users_7d':      set(rng.choice(all_uids, 900, replace=False)),
}

audience_rows = []
for seg_name, uid_set in segments.items():
    for uid in uid_set:
        audience_rows.append({'user_id': uid, 'segment_name': seg_name})
audience_segments = pd.DataFrame(audience_rows)

print('subscriptions shape:', subscriptions.shape)
print('audience_segments shape:', audience_segments.shape)
print('Segment sizes:', {k: len(v) for k,v in segments.items()})


---
## Section 3 — Exploratory Data Analysis

Quick orientation before diving into interview questions.


In [ ]:
# ── Dataset overview ──────────────────────────────────────────────────────────
tables = {
    'users': users, 'app_events': app_events, 'campaigns': campaigns,
    'ad_impressions': ad_impressions, 'ad_clicks': ad_clicks,
    'installs': installs, 'purchases': purchases,
    'subscriptions': subscriptions, 'audience_segments': audience_segments
}
print(f'{"Table":<22} {"Rows":>10} {"Columns":>10}')
print('-' * 44)
for name, df in tables.items():
    print(f'{name:<22} {len(df):>10,} {df.shape[1]:>10}')


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('ShopFlash — Dataset EDA Overview', fontsize=16, fontweight='bold')

# 1. Event frequency
clean_events = app_events[app_events['event_name'].str.islower()].copy()
ev_counts = clean_events['event_name'].value_counts().head(12)
axes[0,0].barh(ev_counts.index, ev_counts.values, color=sns.color_palette('husl', 12))
axes[0,0].set_title('Event Frequency (clean events)')
axes[0,0].set_xlabel('Count')

# 2. Platform split
users['platform'].value_counts().plot.pie(ax=axes[0,1], autopct='%1.1f%%', startangle=90)
axes[0,1].set_title('Users by Platform')
axes[0,1].set_ylabel('')

# 3. Installs by source
installs['install_source'].value_counts().plot.bar(ax=axes[0,2], color=['#2196F3','#4CAF50'])
axes[0,2].set_title('Installs by Source')
axes[0,2].set_xlabel('')
axes[0,2].tick_params(axis='x', rotation=0)

# 4. Revenue distribution
rev_clean = purchases['revenue_usd'].dropna()
rev_clean[rev_clean < 200].hist(bins=40, ax=axes[1,0], color='#FF6B6B', edgecolor='white')
axes[1,0].set_title('Purchase Revenue Distribution (< $200)')
axes[1,0].set_xlabel('Revenue (USD)')

# 5. Daily installs trend
installs_ts = installs.copy()
installs_ts['install_date'] = pd.to_datetime(installs_ts['install_time']).dt.date
daily_installs = installs_ts.groupby('install_date').size()
daily_installs.plot(ax=axes[1,1], color='#4CAF50', linewidth=1.5)
axes[1,1].set_title('Daily Installs Over Time')
axes[1,1].set_xlabel('')

# 6. Campaign budget vs quality
axes[1,2].scatter(campaigns['budget_usd'], campaigns['quality_score'],
                  s=100, c=range(N_CAMPAIGNS), cmap='husl', alpha=0.8)
for _, row in campaigns.iterrows():
    axes[1,2].annotate(row['source'], (row['budget_usd'], row['quality_score']),
                       fontsize=7, alpha=0.7)
axes[1,2].set_title('Campaign Budget vs Traffic Quality')
axes[1,2].set_xlabel('Budget (USD)')
axes[1,2].set_ylabel('Quality Score')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('EDA charts rendered.')


---
## Section 4 — Python Interview Questions

> Questions progress from **Easy → Medium → Hard → Expert**.
> For each question: read the problem, attempt the solution, then review the ideal answer.


---
### Q1 — Event Deduplication & Data Cleaning
**Difficulty:** Easy-Medium | **Time:** 8 min

**Business Context:**  
Your event pipeline occasionally fires duplicate events due to client-side retry logic and SDK bugs. Before any analysis, you must deduplicate the `app_events` table.

**Problem Statement:**  
1. Identify and count true duplicates (same `user_id`, `event_name`, and `event_timestamp` within a 30-second window).  
2. Remove them, keeping the first occurrence.  
3. Also standardize inconsistent event names (e.g., `'App_Open'` → `'app_open'`).  
4. Report how many rows were removed and the final clean count.

**Constraints:**  
- Do NOT drop rows with null timestamps — treat them separately.  
- Preserve the original index after deduplication.

**Follow-up:** How would you handle this at 1B events/day in a production pipeline?

**Common Mistakes:**  
- Dropping on `event_id` only (event_ids on dup rows are intentionally different).  
- Forgetting to handle the null timestamp rows before dedup.  
- Using `.drop_duplicates()` without sorting first (non-deterministic which row is kept).


In [ ]:
# ── Q1 Solution ───────────────────────────────────────────────────────────────

# Step 1: Standardize event names (lowercase + strip)
raw_count = len(app_events)
df = app_events.copy()
df['event_name'] = df['event_name'].str.lower().str.strip().str.replace(' ', '_', regex=False)

# Step 2: Separate null-timestamp rows (cannot deduplicate without a timestamp)
df_null_ts  = df[df['event_timestamp'].isna()].copy()
df_valid_ts = df[df['event_timestamp'].notna()].copy()

print(f'Raw events: {raw_count:,}')
print(f'  Null timestamp rows: {len(df_null_ts):,}')
print(f'  Valid timestamp rows: {len(df_valid_ts):,}')

# Step 3: Coerce timestamps to UTC datetime (mixed tz / tz-naive rows)
def safe_utc(ts):
    if pd.isna(ts): return pd.NaT
    if isinstance(ts, str): ts = pd.Timestamp(ts)
    if ts.tzinfo is None: return ts.tz_localize('UTC')
    return ts.tz_convert('UTC')

df_valid_ts['event_timestamp'] = df_valid_ts['event_timestamp'].apply(safe_utc)

# Step 4: Sort → deduplicate on (user_id, event_name, rounded_timestamp)
# Round to 30-second buckets to catch near-duplicates from retry jitter
df_valid_ts = df_valid_ts.sort_values(['user_id','event_name','event_timestamp'])
df_valid_ts['ts_bucket'] = df_valid_ts['event_timestamp'].dt.floor('30s')

before_dedup = len(df_valid_ts)
df_deduped = df_valid_ts.drop_duplicates(
    subset=['user_id', 'event_name', 'ts_bucket'],
    keep='first'
).drop(columns=['ts_bucket'])

duplicates_removed = before_dedup - len(df_deduped)

# Step 5: Recombine clean events + null-timestamp rows
app_events_clean = pd.concat([df_deduped, df_null_ts], ignore_index=True)

print(f'\nDeduplication Report')
print(f'  Duplicates removed (30s window): {duplicates_removed:,}')
print(f'  Null-timestamp rows (kept aside): {len(df_null_ts):,}')
print(f'  Final clean event count: {len(app_events_clean):,}')
print(f'  Reduction: {duplicates_removed/raw_count*100:.2f}%')

# Event name standardization check
print('\nSample standardized event names:')
print(sorted(app_events_clean['event_name'].unique())[:10])


**Business Interpretation:**  
- ~3% duplicate rate is typical for mobile SDKs with retry logic. Always deduplicate before revenue or retention calculations.  
- Null-timestamp rows should be quarantined into a dead-letter table, not silently dropped.  
- At 1B events/day: use Spark with `dropDuplicates()` on a 30-second watermark in a streaming pipeline (Flink/Spark Streaming). Partition by `user_id` to reduce shuffle.

**Interviewer Note:** Candidate should ask "what counts as a duplicate?" before coding. Deduplication policy is a business decision.


---
### Q2 — Event Co-occurrence Correlation Matrix
**Difficulty:** Medium | **Time:** 12 min

**Business Context:**  
The product team wants to understand which in-app events tend to co-occur in the same user session. High correlation between events may reveal implicit user journeys.

**Problem Statement:**  
1. For each user, create a binary presence matrix: did user perform event X at any point (1/0)?  
2. Compute the Pearson correlation matrix across all event types.  
3. Identify the top 5 most positively correlated event pairs.  
4. Visualize as a heatmap.

**Constraints:**  
- Use only the clean events table (`app_events_clean`). Exclude bot users.  
- Exclude events with fewer than 100 distinct users (too sparse).

**Follow-up:** Why might high correlation between `add_to_cart` and `checkout_start` NOT mean causation?


In [ ]:
# ── Q2 Solution ───────────────────────────────────────────────────────────────

# Step 1: Filter — clean events, exclude bots, valid timestamps only
bots = set(users.loc[users['is_bot'], 'user_id'])
evt_q2 = app_events_clean[
    (~app_events_clean['user_id'].isin(bots)) &
    (app_events_clean['event_timestamp'].notna())
].copy()

# Step 2: Keep only events with >= 100 distinct users
event_user_counts = evt_q2.groupby('event_name')['user_id'].nunique()
valid_events = event_user_counts[event_user_counts >= 100].index.tolist()
evt_q2 = evt_q2[evt_q2['event_name'].isin(valid_events)]

print(f'Valid event types (>= 100 users): {len(valid_events)}')
print(valid_events)

# Step 3: Pivot to binary user × event matrix
binary_matrix = (
    evt_q2.groupby(['user_id','event_name'])
    .size()
    .unstack(fill_value=0)
    .clip(upper=1)   # binarize
)

print(f'\nBinary matrix shape: {binary_matrix.shape}')

# Step 4: Pearson correlation
corr_matrix = binary_matrix.corr(method='pearson')

# Step 5: Top 5 correlated pairs (upper triangle, exclude self)
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
pairs = (
    upper_tri.stack()
    .reset_index()
    .rename(columns={'level_0': 'event_a', 'level_1': 'event_b', 0: 'correlation'})
    .sort_values('correlation', ascending=False)
)

print('\nTop 10 most correlated event pairs:')
print(pairs.head(10).to_string(index=False))

# Step 6: Heatmap
fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-0.3, vmax=0.8,
    linewidths=0.5, ax=ax
)
ax.set_title('Event Co-occurrence Correlation Matrix (Pearson)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('event_correlation.png', dpi=120, bbox_inches='tight')
plt.show()


**Business Interpretation:**  
- High `add_to_cart` ↔ `checkout_start` correlation confirms expected funnel behavior — a healthy signal.  
- `video_watch` ↔ `product_view` correlation reveals content-driven discovery paths — invest more in video.  
- High `push_open` ↔ `app_open` correlation could indicate push notifications are the primary re-engagement driver.  

**Causation trap:** `add_to_cart` and `purchase` may correlate simply because *high-intent users do both* — not because cart adds cause purchases. A/B test removing the cart step would isolate causality.

**Interviewer Note:** Strong candidates will mention *spurious correlation from user intent confound* and suggest Granger causality or event sequencing tests.


---
### Q3 — Event Transition Probability Matrix
**Difficulty:** Medium | **Time:** 15 min

**Business Context:**  
The growth team needs a next-event predictor for personalization. First, build an event transition matrix showing P(next_event | current_event) within sessions.

**Problem Statement:**  
1. Within each session, sort events by timestamp.  
2. Build a transition count matrix: for each consecutive event pair (A → B), count occurrences.  
3. Normalize rows to get transition probabilities.  
4. Visualize and report the most likely next event after `product_view`.

**Constraints:**  
- Use clean events, exclude bots, exclude null timestamps.  
- Only consider transitions within 10 minutes of each other (cross-session leakage otherwise).


In [ ]:
# ── Q3 Solution ───────────────────────────────────────────────────────────────

evt_q3 = app_events_clean[
    (~app_events_clean['user_id'].isin(bots)) &
    (app_events_clean['event_timestamp'].notna()) &
    (app_events_clean['event_name'].isin(valid_events))
].copy()

# Ensure timestamps are UTC-aware
evt_q3['event_timestamp'] = evt_q3['event_timestamp'].apply(
    lambda x: x.tz_localize('UTC') if (pd.notna(x) and getattr(x,'tzinfo',None) is None) else x
)

# Sort within session
evt_q3 = evt_q3.sort_values(['session_id','event_timestamp'])

# Build transitions: shift within session only
evt_q3['next_event'] = evt_q3.groupby('session_id')['event_name'].shift(-1)
evt_q3['next_ts']    = evt_q3.groupby('session_id')['event_timestamp'].shift(-1)

# Filter: within 10 minutes of next event, same session
evt_q3['gap_min'] = (
    (evt_q3['next_ts'] - evt_q3['event_timestamp'])
    .dt.total_seconds() / 60
)
transitions = evt_q3[
    (evt_q3['next_event'].notna()) &
    (evt_q3['gap_min'].between(0, 10))
]

# Count matrix
trans_counts = (
    transitions.groupby(['event_name','next_event'])
    .size()
    .unstack(fill_value=0)
)

# Normalize rows → probabilities
trans_prob = trans_counts.div(trans_counts.sum(axis=1), axis=0).round(4)

print('Transition matrix shape:', trans_prob.shape)
print('\nTop next events after product_view:')
if 'product_view' in trans_prob.index:
    print(trans_prob.loc['product_view'].sort_values(ascending=False).head(6))

# Visualize
fig, ax = plt.subplots(figsize=(13, 8))
sns.heatmap(
    trans_prob, annot=True, fmt='.2f', cmap='Blues',
    linewidths=0.3, ax=ax, vmin=0, vmax=0.4
)
ax.set_title('Event Transition Probability Matrix  P(next | current)', fontsize=13, fontweight='bold')
ax.set_xlabel('Next Event')
ax.set_ylabel('Current Event')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('transition_matrix.png', dpi=120, bbox_inches='tight')
plt.show()


**Business Interpretation:**  
- A healthy `product_view → add_to_cart` transition rate of 15–25% indicates strong product-page performance.  
- `product_view → app_close` above 40% is a red flag — user intent drops at the product detail page (pricing, image quality, or missing info).  
- Use this matrix to power a next-event recommendation engine or to personalize push notifications.

**Interviewer Note:** Trap question — ask candidate how they handle the session boundary problem. Naively shifting rows leaks transitions across sessions.


---
### Q4 — Multi-Step Funnel Analysis
**Difficulty:** Medium | **Time:** 12 min

**Business Context:**  
The conversion team wants to know where users drop off in the purchase funnel. Measure the ordered funnel: `app_open → product_view → add_to_cart → checkout_start → purchase`.

**Problem Statement:**  
1. For each user, check if they performed each step *in order* (step N must occur after step N-1, not necessarily on the same day).  
2. Compute conversion rate at each step.  
3. Identify the biggest drop-off step.  
4. Break down conversion rates by platform (iOS vs Android).

**Constraints:**  
- Use clean events, exclude bots.  
- Each step must happen at least once (user can repeat steps).


In [ ]:
# ── Q4 Solution ───────────────────────────────────────────────────────────────
FUNNEL = ['app_open','product_view','add_to_cart','checkout_start','purchase']

evt_q4 = app_events_clean[
    (~app_events_clean['user_id'].isin(bots)) &
    (app_events_clean['event_timestamp'].notna()) &
    (app_events_clean['event_name'].isin(FUNNEL))
].copy()

evt_q4['event_timestamp'] = evt_q4['event_timestamp'].apply(
    lambda x: x.tz_localize('UTC') if (pd.notna(x) and getattr(x,'tzinfo',None) is None) else x
)
evt_q4 = evt_q4.sort_values(['user_id','event_timestamp'])

def funnel_steps_completed(group, funnel):
    """Return the index of the last funnel step completed in order."""
    events = group[['event_name','event_timestamp']].values.tolist()
    step = 0
    for evt_name, ts in events:
        if step < len(funnel) and evt_name == funnel[step]:
            step += 1
        if step == len(funnel):
            break
    return step  # 1 = completed step 1, 5 = full conversion

# Per-user funnel completion
user_funnel = evt_q4.groupby('user_id').apply(funnel_steps_completed, funnel=FUNNEL)
user_funnel = user_funnel.reset_index()
user_funnel.columns = ['user_id','steps_completed']
user_funnel = user_funnel.merge(users[['user_id','platform']], on='user_id', how='left')

# Overall conversion per step
total_top = (user_funnel['steps_completed'] >= 1).sum()
print(f'Overall Funnel (from app_open):')
for i, step in enumerate(FUNNEL, 1):
    count = (user_funnel['steps_completed'] >= i).sum()
    cvr_vs_top = count / total_top * 100
    cvr_vs_prev = count / max((user_funnel['steps_completed'] >= max(i-1,1)).sum(), 1) * 100
    print(f'  Step {i} — {step:<20} {count:>6,} users  ({cvr_vs_top:.1f}% from top | {cvr_vs_prev:.1f}% from prev)')

# Platform breakdown
print('\nPlatform Funnel Comparison:')
for platform in ['iOS','Android']:
    sub = user_funnel[user_funnel['platform'] == platform]
    top = (sub['steps_completed'] >= 1).sum()
    conv = (sub['steps_completed'] >= len(FUNNEL)).sum()
    print(f'  {platform:<10} top={top:,}  full_conversion={conv:,}  rate={conv/max(top,1)*100:.2f}%')

# Visualization
step_counts = [(user_funnel['steps_completed'] >= i).sum() for i in range(1, len(FUNNEL)+1)]
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(FUNNEL, step_counts, color=sns.color_palette('RdYlGn_r', len(FUNNEL)))
for bar, count in zip(bars, step_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{count:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Purchase Funnel — Users Reaching Each Step', fontsize=13, fontweight='bold')
ax.set_ylabel('Users')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('funnel.png', dpi=120, bbox_inches='tight')
plt.show()


**Business Interpretation:**  
- The biggest drop-off is typically `add_to_cart → checkout_start` in commerce apps (~40–60% drop).  
- iOS users typically convert at higher rates (higher avg income, more trust in in-app payments).  
- Android drop-off at checkout often indicates payment friction (3DS, UPI failures in emerging markets).

**Interviewer Note:** Watch for candidates who don't enforce *event ordering* — counting users who ever did each event (regardless of order) inflates conversion rates.


---
### Q5 — Paid vs Organic Funnel Comparison
**Difficulty:** Medium | **Time:** 10 min

**Business Context:**  
Marketing claims paid users convert better than organic. Validate this claim using funnel data, and check whether the difference is statistically significant.

**Problem Statement:**  
1. Tag each user as `paid` or `organic` based on their `acquisition_source`.  
2. Compare full-funnel conversion rates (app_open → purchase) for each group.  
3. Run a chi-squared test to determine if the difference is statistically significant.

**Follow-up (trap):** The paid group converts at 8% vs organic at 4%. Does this mean paid acquisition is more valuable? What confound might explain this?


In [ ]:
# ── Q5 Solution ───────────────────────────────────────────────────────────────

paid_sources = {'facebook','google','tiktok','apple','email','influencer'}
users['acq_type'] = users['acquisition_source'].apply(
    lambda s: 'paid' if s in paid_sources else 'organic'
)

# Merge acquisition type onto funnel data
user_funnel_acq = user_funnel.merge(users[['user_id','acq_type']], on='user_id', how='left')

print('Paid vs Organic Funnel Conversion:')
print('-' * 50)
summary_rows = []
for acq in ['paid','organic']:
    sub = user_funnel_acq[user_funnel_acq['acq_type'] == acq]
    top  = (sub['steps_completed'] >= 1).sum()
    conv = (sub['steps_completed'] == len(FUNNEL)).sum()
    rate = conv / max(top, 1)
    summary_rows.append({'acq_type': acq, 'users_at_top': top, 'converters': conv, 'cvr': rate})
    print(f'  {acq:<10}  users={top:,}  converters={conv:,}  CVR={rate*100:.2f}%')

# Chi-squared test
paid_row    = [r for r in summary_rows if r['acq_type'] == 'paid'][0]
org_row     = [r for r in summary_rows if r['acq_type'] == 'organic'][0]

paid_conv   = paid_row['converters']
paid_noconv = paid_row['users_at_top'] - paid_conv
org_conv    = org_row['converters']
org_noconv  = org_row['users_at_top'] - org_conv

contingency = np.array([[paid_conv, paid_noconv], [org_conv, org_noconv]])
chi2, p_val, dof, expected = chi2_contingency(contingency)

print(f'\nChi-Squared Test:')
print(f'  chi2={chi2:.4f}  p-value={p_val:.6f}  dof={dof}')
if p_val < 0.05:
    print('  ✓ Statistically significant difference (p < 0.05)')
else:
    print('  ✗ Difference NOT statistically significant')

print('\n[TRAP] Answer: Paid users may appear to convert better because:')
print('  1. They were targeted by intent-based campaigns (already high purchase intent)')
print('  2. Paid campaigns often retarget existing users who already browsed')
print('  3. Selection bias — organic users include casual browsers')
print('  → Correct comparison: incrementality test with holdout group')


---
### Q6 — Multi-Touch Attribution Analysis
**Difficulty:** Medium-Hard | **Time:** 18 min

**Business Context:**  
ShopFlash uses last-click attribution by default via AppsFlyer. The CMO suspects this over-credits bottom-funnel channels (retargeting) and under-credits upper-funnel channels (brand awareness). Implement and compare 3 attribution models.

**Problem Statement:**  
For each purchase, look back 7 days for ad clicks from the same user.  
Implement:  
1. **Last-click** — full credit to the last touchpoint  
2. **First-click** — full credit to the first touchpoint  
3. **Linear** — equal credit split across all touchpoints  

Report credit (in $) allocated to each campaign under each model.

**Constraints:**  
- Attribution window: 7 days before purchase.  
- Use `purchases` table joined with `ad_clicks`.


In [ ]:
# ── Q6 Solution ───────────────────────────────────────────────────────────────

# Prepare: coerce timestamps
purch = purchases[purchases['revenue_usd'].notna()].copy()
purch['purchase_time'] = purch['purchase_time'].dt.tz_localize('UTC', ambiguous='NaT', nonexistent='NaT')

clks = ad_clicks.copy()
clks['click_time'] = pd.to_datetime(clks['click_time'])
if clks['click_time'].dt.tz is None:
    clks['click_time'] = clks['click_time'].dt.tz_localize('UTC')

# Cross-join purchase × clicks for same user within 7 days
# Use merge_asof trick: merge on user_id then filter time window
purch_small = purch[['purchase_id','user_id','purchase_time','revenue_usd']]
clks_small  = clks[['click_id','user_id','click_time','campaign_id']]

joined = purch_small.merge(clks_small, on='user_id', how='left')
joined['days_before_purchase'] = (
    (joined['purchase_time'] - joined['click_time'])
    .dt.total_seconds() / 86_400
)

# Filter: click must be within 7 days before purchase
attr_window = joined[
    (joined['days_before_purchase'].between(0, 7)) &
    (joined['campaign_id'].notna())
].copy()

print(f'Purchases with at least 1 touchpoint: {attr_window["purchase_id"].nunique():,}')
print(f'Total purchase-touchpoint pairs: {len(attr_window):,}')

# ── Last-Click ────────────────────────────────────────────────────────────────
last_click = (
    attr_window.sort_values('click_time')
    .groupby('purchase_id')
    .last()
    .reset_index()[['purchase_id','campaign_id','revenue_usd']]
)
last_click_rev = last_click.groupby('campaign_id')['revenue_usd'].sum().rename('last_click_rev')

# ── First-Click ───────────────────────────────────────────────────────────────
first_click = (
    attr_window.sort_values('click_time')
    .groupby('purchase_id')
    .first()
    .reset_index()[['purchase_id','campaign_id','revenue_usd']]
)
first_click_rev = first_click.groupby('campaign_id')['revenue_usd'].sum().rename('first_click_rev')

# ── Linear ────────────────────────────────────────────────────────────────────
attr_window['n_touchpoints'] = attr_window.groupby('purchase_id')['click_id'].transform('count')
attr_window['linear_credit']  = attr_window['revenue_usd'] / attr_window['n_touchpoints']
linear_rev = attr_window.groupby('campaign_id')['linear_credit'].sum().rename('linear_rev')

# ── Compare models ────────────────────────────────────────────────────────────
attr_compare = (
    pd.DataFrame([last_click_rev, first_click_rev, linear_rev])
    .T
    .fillna(0)
    .round(2)
    .sort_values('last_click_rev', ascending=False)
)
attr_compare = attr_compare.merge(campaigns[['campaign_id','campaign_name','source']], left_index=True, right_on='campaign_id')

print('\nAttribution Comparison (Revenue $):')
print(attr_compare[['campaign_name','source','last_click_rev','first_click_rev','linear_rev']].to_string(index=False))

# Visualization
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(attr_compare))
w = 0.28
ax.bar(x - w, attr_compare['last_click_rev'],  w, label='Last-Click',  color='#E74C3C')
ax.bar(x,     attr_compare['first_click_rev'], w, label='First-Click', color='#3498DB')
ax.bar(x + w, attr_compare['linear_rev'],      w, label='Linear',      color='#2ECC71')
ax.set_xticks(x)
ax.set_xticklabels(attr_compare['campaign_name'], rotation=45, ha='right', fontsize=9)
ax.set_title('Attribution Model Comparison by Campaign', fontsize=13, fontweight='bold')
ax.set_ylabel('Revenue Attributed ($)')
ax.legend()
plt.tight_layout()
plt.savefig('attribution.png', dpi=120, bbox_inches='tight')
plt.show()


**Business Interpretation:**  
- **Retargeting campaigns** (CMP_003) receive disproportionate last-click credit because they fire late in the funnel — they intercept high-intent users who would have converted anyway.  
- **Brand awareness campaigns** (CMP_001, CMP_015) gain credit under first-click/linear models, better reflecting their role in initiating the funnel.  
- **Recommendation:** Move to data-driven attribution (Shapley values or Markov chains) for budget optimization. Linear model is a safe interim step.

**Interviewer Note:** Top candidates will mention *attribution inflation* — if the same purchase gets attributed to multiple campaigns under different models, the sum of attributed revenue can exceed actual revenue.


---
### Q7 — D1 / D7 / D30 Cohort Retention Analysis
**Difficulty:** Hard | **Time:** 20 min

**Business Context:**  
Retention is the north-star metric for ShopFlash. Build a cohort retention table showing what % of users who installed in week W came back on Day 1, Day 7, and Day 30.

**Problem Statement:**  
1. Define a user's install cohort as the week of their install date.  
2. For each cohort, calculate D1, D7, and D30 retention rates.  
3. Retention = user had at least one `app_open` event on day N (±1 day tolerance).  
4. Visualize as a cohort heatmap.

**Constraints:**  
- Exclude bot users. Only include cohorts with 50+ users.


In [ ]:
# ── Q7 Solution ───────────────────────────────────────────────────────────────

# User install dates (clean, UTC)
user_installs = users[~users['is_bot']][['user_id','install_date']].copy()
user_installs['install_date'] = pd.to_datetime(user_installs['install_date']).dt.tz_localize(
    'UTC', ambiguous='NaT', nonexistent='NaT'
)
user_installs['cohort_week'] = user_installs['install_date'].dt.to_period('W').dt.start_time

# App_open events for non-bots
app_opens = app_events_clean[
    (app_events_clean['event_name'] == 'app_open') &
    (~app_events_clean['user_id'].isin(bots)) &
    (app_events_clean['event_timestamp'].notna())
][['user_id','event_timestamp']].copy()

app_opens['event_timestamp'] = app_opens['event_timestamp'].apply(
    lambda x: x if getattr(x,'tzinfo',None) else pd.Timestamp(x, tz='UTC')
)
app_opens['event_date'] = app_opens['event_timestamp'].dt.normalize()

# Merge app_opens with install dates
retention_df = app_opens.merge(user_installs[['user_id','install_date','cohort_week']], on='user_id', how='inner')
retention_df['days_since_install'] = (
    (retention_df['event_date'] - retention_df['install_date'])
    .dt.days
)

def cohort_retention(df, day_target, tolerance=1):
    """Return retention rate for day_target±tolerance."""
    day_df = df[df['days_since_install'].between(day_target - tolerance, day_target + tolerance)]
    retained = day_df.groupby('cohort_week')['user_id'].nunique().rename(f'd{day_target}_retained')
    return retained

cohort_sizes = user_installs.groupby('cohort_week')['user_id'].count().rename('cohort_size')
cohort_sizes = cohort_sizes[cohort_sizes >= 50]

d1  = cohort_retention(retention_df, 1)
d7  = cohort_retention(retention_df, 7)
d30 = cohort_retention(retention_df, 30)

cohort_table = pd.DataFrame({'cohort_size': cohort_sizes})
cohort_table = cohort_table.join(d1).join(d7).join(d30)
cohort_table['d1_rate']  = (cohort_table['d1_retained']  / cohort_table['cohort_size']).round(4)
cohort_table['d7_rate']  = (cohort_table['d7_retained']  / cohort_table['cohort_size']).round(4)
cohort_table['d30_rate'] = (cohort_table['d30_retained'] / cohort_table['cohort_size']).round(4)
cohort_table = cohort_table.fillna(0)

print('Cohort Retention Table:')
print(cohort_table[['cohort_size','d1_rate','d7_rate','d30_rate']].to_string())

# Heatmap
retention_heat = cohort_table[['d1_rate','d7_rate','d30_rate']].copy()
retention_heat.index = retention_heat.index.astype(str)
retention_heat.columns = ['D1','D7','D30']

fig, ax = plt.subplots(figsize=(8, max(6, len(retention_heat)//2)))
sns.heatmap(
    retention_heat * 100, annot=True, fmt='.1f', cmap='YlOrRd_r',
    vmin=0, vmax=50, linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Retention %'}
)
ax.set_title('Cohort Retention Heatmap (D1 / D7 / D30)', fontsize=13, fontweight='bold')
ax.set_xlabel('Retention Day')
ax.set_ylabel('Install Cohort Week')
plt.tight_layout()
plt.savefig('retention_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()


**Business Interpretation:**  
- Typical mobile app benchmarks: D1=25-40%, D7=10-20%, D30=5-10%. Below benchmark signals onboarding problems.  
- Cohorts from paid campaigns showing higher D1 but steeper D30 drop-off may indicate low-quality traffic (bot or bored users).  
- Improving D1 retention by 5pp can increase D30 by 2-3pp, significantly impacting LTV.

**Common Mistakes:**  
- Using `event_date == install_date + timedelta(days=N)` without tolerance → misses users who log in slightly off-day.  
- Not normalizing by cohort size → larger cohorts dominate the metric.  
- Including users with installs so recent they can't have D30 data yet (denominator inflation).


---
### Q8 — Sessionization from Raw Event Stream
**Difficulty:** Medium-Hard | **Time:** 15 min

**Business Context:**  
The existing `session_id` in the dataset was assigned by the client SDK, which has known bugs causing session splits and merges. Rebuild sessions server-side from the event stream using a 30-minute inactivity timeout.

**Problem Statement:**  
1. For each user, sort events by timestamp.  
2. Mark a new session start whenever the gap to the previous event exceeds 30 minutes.  
3. Assign new `server_session_id` values.  
4. Compute per-session statistics: duration, event count, events in session.

**Constraints:**  
- Only use events with valid timestamps and non-bot users.


In [ ]:
# ── Q8 Solution ───────────────────────────────────────────────────────────────

evt_sess = app_events_clean[
    (~app_events_clean['user_id'].isin(bots)) &
    (app_events_clean['event_timestamp'].notna())
][['user_id','event_name','event_timestamp']].copy()

evt_sess['event_timestamp'] = evt_sess['event_timestamp'].apply(
    lambda x: x.tz_localize('UTC') if (pd.notna(x) and getattr(x,'tzinfo',None) is None) else x
)
evt_sess = evt_sess.sort_values(['user_id','event_timestamp']).reset_index(drop=True)

# Time gap since previous event for same user
evt_sess['prev_ts']  = evt_sess.groupby('user_id')['event_timestamp'].shift(1)
evt_sess['gap_min']  = (
    (evt_sess['event_timestamp'] - evt_sess['prev_ts'])
    .dt.total_seconds() / 60
)

# New session = first event of user OR gap > 30 min
evt_sess['is_new_session'] = (
    evt_sess['gap_min'].isna() |         # first event of user
    (evt_sess['gap_min'] > SESSION_TIMEOUT)
)

# Cumulative sum gives session number per user
evt_sess['session_num'] = evt_sess.groupby('user_id')['is_new_session'].cumsum()
evt_sess['server_session_id'] = evt_sess['user_id'] + '_S' + evt_sess['session_num'].astype(str).str.zfill(4)

# Session stats
session_stats = evt_sess.groupby('server_session_id').agg(
    user_id        = ('user_id', 'first'),
    session_start  = ('event_timestamp', 'min'),
    session_end    = ('event_timestamp', 'max'),
    event_count    = ('event_name', 'count'),
    events_in_sess = ('event_name', lambda x: ', '.join(sorted(x.unique())))
).reset_index()

session_stats['duration_min'] = (
    (session_stats['session_end'] - session_stats['session_start'])
    .dt.total_seconds() / 60
).round(2)

print(f'Total server-side sessions: {len(session_stats):,}')
print(f'Avg events per session: {session_stats["event_count"].mean():.2f}')
print(f'Median session duration: {session_stats["duration_min"].median():.2f} min')
print(f'Sessions with 1 event: {(session_stats["event_count"]==1).sum():,} (bounces)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

session_stats['event_count'].clip(upper=20).hist(bins=20, ax=axes[0], color='#3498DB', edgecolor='white')
axes[0].set_title('Events per Session Distribution')
axes[0].set_xlabel('Event Count')

session_stats['duration_min'].clip(upper=60).hist(bins=30, ax=axes[1], color='#E67E22', edgecolor='white')
axes[1].set_title('Session Duration Distribution (≤60 min)')
axes[1].set_xlabel('Duration (minutes)')

plt.tight_layout()
plt.savefig('sessions.png', dpi=120, bbox_inches='tight')
plt.show()


**Business Interpretation:**  
- High single-event session % = poor onboarding or content relevance issues.  
- Server-side sessionization often produces 15–30% more sessions than client-side (SDK merges sessions across app minimizes).  
- Sessions > 45 minutes are likely background activity (app not closed) — consider adding a max session cap (e.g., 4 hours).

**Interviewer Note:** Trap — `cumsum()` only works correctly when sorted by `(user_id, event_timestamp)`. If candidates don't sort first, session IDs will be wrong across rows.


---
### Q9 — Campaign Quality Scoring (CAC, CTR, CVR, ROAS)
**Difficulty:** Medium | **Time:** 12 min

**Business Context:**  
The UA team needs a single dashboard view of campaign efficiency across all 15 campaigns.

**Problem Statement:**  
For each campaign, compute:  
1. **Impressions** — total ad impressions  
2. **Clicks** — total ad clicks  
3. **CTR** = Clicks / Impressions  
4. **Installs** — attributed installs  
5. **CVR** = Installs / Clicks  
6. **Spend** = sum of click costs + impression costs  
7. **CAC** = Spend / Installs  
8. **Revenue** = sum of purchase revenue attributed to campaign  
9. **ROAS** = Revenue / Spend  
10. **Quality Score** = composite (penalize high CAC, low retention signal)

Rank campaigns and flag underperformers.


In [ ]:
# ── Q9 Solution ───────────────────────────────────────────────────────────────

# Impressions per campaign
impr_agg = ad_impressions.groupby('campaign_id').agg(
    impressions = ('impression_id','count'),
    impr_cost   = ('cost_usd','sum')
).reset_index()

# Clicks per campaign
click_agg = ad_clicks.groupby('campaign_id').agg(
    clicks     = ('click_id','count'),
    click_cost = ('cost_usd','sum')
).reset_index()

# Installs per campaign (exclude bots)
install_agg = installs[
    (~installs['is_bot_install']) & installs['campaign_id'].notna()
].groupby('campaign_id').agg(
    installs = ('install_id','count')
).reset_index()

# Revenue per campaign
rev_agg = purchases[
    purchases['revenue_usd'].notna() & purchases['campaign_attributed'].notna()
].groupby('campaign_attributed').agg(
    revenue = ('revenue_usd','sum')
).reset_index().rename(columns={'campaign_attributed':'campaign_id'})

# Merge all
camp_perf = campaigns[['campaign_id','campaign_name','source','budget_usd']].copy()
camp_perf = (
    camp_perf
    .merge(impr_agg,   on='campaign_id', how='left')
    .merge(click_agg,  on='campaign_id', how='left')
    .merge(install_agg,on='campaign_id', how='left')
    .merge(rev_agg,    on='campaign_id', how='left')
    .fillna(0)
)

camp_perf['spend']    = camp_perf['impr_cost'] + camp_perf['click_cost']
camp_perf['CTR']      = np.where(camp_perf['impressions']>0, camp_perf['clicks']/camp_perf['impressions'], 0)
camp_perf['CVR']      = np.where(camp_perf['clicks']>0,      camp_perf['installs']/camp_perf['clicks'], 0)
camp_perf['CAC']      = np.where(camp_perf['installs']>0,    camp_perf['spend']/camp_perf['installs'], 0)
camp_perf['ROAS']     = np.where(camp_perf['spend']>0,       camp_perf['revenue']/camp_perf['spend'], 0)

# Simple composite quality score (normalize 0-1, higher = better)
for col, ascending in [('CAC', True), ('ROAS', False), ('CVR', False), ('CTR', False)]:
    mn, mx = camp_perf[col].min(), camp_perf[col].max()
    if mx > mn:
        norm = (camp_perf[col] - mn) / (mx - mn)
        camp_perf[f'{col}_norm'] = norm if not ascending else 1 - norm
    else:
        camp_perf[f'{col}_norm'] = 0.5

camp_perf['composite_score'] = (
    0.35 * camp_perf['ROAS_norm'] +
    0.30 * camp_perf['CAC_norm']  +
    0.20 * camp_perf['CVR_norm']  +
    0.15 * camp_perf['CTR_norm']
).round(4)

cols_show = ['campaign_name','source','impressions','clicks','installs',
             'spend','CAC','revenue','ROAS','CTR','CVR','composite_score']
result = camp_perf[cols_show].sort_values('composite_score', ascending=False)

print('Campaign Performance Dashboard:')
print(result.to_string(index=False, float_format='{:.4f}'.format))

# Flag underperformers
thresh = result['composite_score'].quantile(0.33)
print(f'\nUnderperforming campaigns (bottom 33%, score < {thresh:.3f}):')
print(result[result['composite_score'] < thresh][['campaign_name','source','composite_score']])


**Business Interpretation:**  
- **Retargeting (CMP_003)** and **Apple Search Ads (CMP_010)** typically score highest — high intent, high CVR.  
- **Influencer (CMP_006)** and **YouTube Brand (CMP_014)** score lowest on direct ROAS — but may drive organic tail. Don't kill them based on last-click alone.  
- ROAS > 3x is a healthy breakeven for most consumer apps with 30-40% margin.  
- CAC should be compared against D90 LTV, not just first-purchase revenue.

**Interviewer Note:** Strong candidates will ask *"should I include view-through conversions in ROAS?"* — this is a key data governance question in attribution.


---
### Q10 — Bot & Fraud Traffic Detection
**Difficulty:** Hard | **Time:** 20 min

**Business Context:**  
Your AppsFlyer implementation flagged 5% of installs as bot traffic, but the data science team suspects the actual fraud rate is higher. Use behavioral signals to identify additional suspicious users.

**Problem Statement:**  
Build a fraud detection pipeline that identifies bot-like users based on:  
1. **Install velocity** — many installs from the same device_id in a short window  
2. **Event velocity** — unusually high event rate per hour  
3. **Behavioral anomaly** — IsolationForest on behavioral features  
4. **Impossible events** — events timestamped before install  

Combine signals into a fraud score and flag high-risk users.


In [ ]:
# ── Q10 Solution ───────────────────────────────────────────────────────────────

# Signal 1: Install velocity per device_id (>3 installs from same device = suspicious)
device_install_count = installs.groupby('device_id')['install_id'].count().rename('device_install_count')

# Signal 2: Event velocity per user (events per active hour)
evt_vel = app_events_clean[
    app_events_clean['event_timestamp'].notna()
][['user_id','event_timestamp']].copy()
evt_vel['event_timestamp'] = evt_vel['event_timestamp'].apply(
    lambda x: x.tz_localize('UTC') if (pd.notna(x) and getattr(x,'tzinfo',None) is None) else x
)

user_event_stats = evt_vel.groupby('user_id').agg(
    total_events   = ('event_timestamp','count'),
    first_event    = ('event_timestamp','min'),
    last_event     = ('event_timestamp','max')
).reset_index()
user_event_stats['active_hours'] = (
    (user_event_stats['last_event'] - user_event_stats['first_event'])
    .dt.total_seconds() / 3600
).clip(lower=0.1)
user_event_stats['events_per_hour'] = user_event_stats['total_events'] / user_event_stats['active_hours']

# Signal 3: Impossible events (event before install date)
user_with_install = users[['user_id','install_date']].copy()
user_with_install['install_date'] = pd.to_datetime(user_with_install['install_date']).apply(
    lambda x: x.tz_localize('UTC') if (pd.notna(x) and getattr(x,'tzinfo',None) is None) else x
)
impossible = evt_vel.merge(user_with_install, on='user_id', how='left')
impossible_count = (
    impossible[impossible['event_timestamp'] < impossible['install_date']]
    .groupby('user_id')
    .size()
    .rename('impossible_events')
)

# Combine signals into user-level fraud features
fraud_features = users[['user_id','device_id','is_bot']].merge(
    device_install_count, on='device_id', how='left'
).merge(
    user_event_stats[['user_id','total_events','events_per_hour']], on='user_id', how='left'
).merge(
    impossible_count, on='user_id', how='left'
).fillna(0)

# Signal 4: IsolationForest on behavioral features
feature_cols = ['device_install_count','events_per_hour','total_events','impossible_events']
X = fraud_features[feature_cols].fillna(0).values
Scaler = StandardScaler()
X_scaled = Scaler.fit_transform(X)

isoforest = IsolationForest(contamination=0.08, random_state=42)
fraud_features['anomaly_score'] = isoforest.fit_predict(X_scaled)  # -1 = anomaly
fraud_features['if_fraud']      = (fraud_features['anomaly_score'] == -1).astype(int)

# Composite fraud score (rule-based + model)
fraud_features['rule_fraud'] = (
    (fraud_features['device_install_count'] > 3).astype(int) +
    (fraud_features['events_per_hour'] > 200).astype(int) +
    (fraud_features['impossible_events'] > 0).astype(int)
)
fraud_features['fraud_score'] = (
    0.4 * fraud_features['if_fraud'] +
    0.4 * (fraud_features['rule_fraud'] > 0).astype(int) +
    0.2 * fraud_features['is_bot'].astype(int)
).round(2)
fraud_features['is_flagged'] = fraud_features['fraud_score'] >= 0.5

print('Fraud Detection Summary:')
print(f'  Known bots (AppsFlyer):   {fraud_features["is_bot"].sum():,}')
print(f'  IsolationForest flagged:   {fraud_features["if_fraud"].sum():,}')
print(f'  Rule-based flagged:        {(fraud_features["rule_fraud"] > 0).sum():,}')
print(f'  Combined high-risk:        {fraud_features["is_flagged"].sum():,}')
print(f'  Estimated true fraud rate: {fraud_features["is_flagged"].mean()*100:.1f}%')

# Distribution of fraud scores
fig, ax = plt.subplots(figsize=(10, 4))
fraud_features['fraud_score'].hist(bins=20, ax=ax, color='#E74C3C', edgecolor='white')
ax.axvline(0.5, color='black', linestyle='--', label='Flag Threshold')
ax.set_title('User Fraud Score Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Fraud Score')
ax.legend()
plt.tight_layout()
plt.savefig('fraud_scores.png', dpi=120, bbox_inches='tight')
plt.show()


**Business Interpretation:**  
- True fraud rate is often 2–3× the SDK-flagged rate. Financial impact: if 10% of your $400K/month spend goes to fraud, that's $40K/month wasted.  
- Impossible events (event before install) are definitive fraud signals — always check this.  
- IsolationForest works well for unsupervised fraud detection but requires tuning `contamination` against labeled data.

**Interviewer Note:** Ask candidates how they'd handle false positives — blocking legitimate users is costly. Suggest a tiered approach: low-score → monitor, medium → throttle, high → block.


---
### Q11 — LTV Estimation by Acquisition Cohort
**Difficulty:** Hard | **Time:** 20 min

**Business Context:**  
Finance needs a 90-day LTV estimate per acquisition source to set optimal CAC targets for next quarter's UA budget.

**Problem Statement:**  
1. For each user, compute cumulative revenue over 30, 60, and 90 days post-install.  
2. Group by acquisition source and report average LTV at each milestone.  
3. Fit a simple power-law curve to estimate D180 LTV.
4. Flag any source where CAC > D90 LTV (unprofitable).

**Constraints:**  
- Exclude bot users and refunded purchases.  
- Only include cohorts with sufficient 90-day observation windows (installed before April 1, 2024).


In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from scipy.stats import ttest_ind, chi2_contingency, mannwhitneyu
from collections import Counter, defaultdict
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

# NOTE: In the combined notebook these datasets are already loaded.
# For standalone execution, re-run Part 1 first.
# Here we demonstrate the logic assuming the dataframes exist.
print('Q11: LTV Estimation by Acquisition Cohort')
print('=' * 50)

# Simulating required dataframes if running standalone
try:
    _ = users
except NameError:
    print('WARNING: Run Part 1 first to load all datasets.')
    raise

# Step 1: Filter eligible users
cutoff_date = pd.Timestamp('2024-04-01', tz='UTC')
bots_set = set(users.loc[users['is_bot'], 'user_id'])

eligible_users = users[
    (~users['is_bot']) &
    (pd.to_datetime(users['install_date']).dt.tz_localize('UTC', ambiguous='NaT', nonexistent='NaT') < cutoff_date)
][['user_id','acquisition_source','install_date']].copy()
eligible_users['install_date'] = pd.to_datetime(eligible_users['install_date']).dt.tz_localize('UTC', ambiguous='NaT', nonexistent='NaT')

# Step 2: Clean purchases (no refunds, no missing revenue)
purch_clean = purchases[
    (~purchases['is_refunded']) &
    purchases['revenue_usd'].notna()
][['user_id','purchase_time','revenue_usd']].copy()
purch_clean['purchase_time'] = pd.to_datetime(purch_clean['purchase_time'])
if purch_clean['purchase_time'].dt.tz is None:
    purch_clean['purchase_time'] = purch_clean['purchase_time'].dt.tz_localize('UTC')

# Step 3: Join purchases with eligible users
ltv_df = purch_clean.merge(eligible_users, on='user_id', how='inner')
ltv_df['days_post_install'] = (
    (ltv_df['purchase_time'] - ltv_df['install_date'])
    .dt.total_seconds() / 86_400
)
if ltv_df.empty:
    print('No matching data — using synthetic LTV for demonstration')
else:
    ltv_30  = ltv_df[ltv_df['days_post_install'] <= 30].groupby(['user_id','acquisition_source'])['revenue_usd'].sum().reset_index().rename(columns={'revenue_usd':'ltv_30'})
    ltv_60  = ltv_df[ltv_df['days_post_install'] <= 60].groupby(['user_id','acquisition_source'])['revenue_usd'].sum().reset_index().rename(columns={'revenue_usd':'ltv_60'})
    ltv_90  = ltv_df[ltv_df['days_post_install'] <= 90].groupby(['user_id','acquisition_source'])['revenue_usd'].sum().reset_index().rename(columns={'revenue_usd':'ltv_90'})

    user_ltv = eligible_users.merge(ltv_30, on=['user_id','acquisition_source'], how='left')
    user_ltv = user_ltv.merge(ltv_60[['user_id','ltv_60']], on='user_id', how='left')
    user_ltv = user_ltv.merge(ltv_90[['user_id','ltv_90']], on='user_id', how='left')
    user_ltv[['ltv_30','ltv_60','ltv_90']] = user_ltv[['ltv_30','ltv_60','ltv_90']].fillna(0)

    ltv_by_source = user_ltv.groupby('acquisition_source')[['ltv_30','ltv_60','ltv_90']].mean().round(2)
    print('\nAverage LTV by Acquisition Source:')
    print(ltv_by_source)

# Synthetic demo if data is sparse
sources_demo = ['facebook','google','tiktok','organic','referral','email','apple']
ltv_demo = pd.DataFrame({
    'source':  sources_demo,
    'ltv_30':  [18.5, 22.0, 12.0, 28.0, 35.0, 8.0, 25.0],
    'ltv_60':  [28.0, 33.0, 18.0, 40.0, 50.0, 12.0, 38.0],
    'ltv_90':  [34.0, 40.0, 22.0, 48.0, 58.0, 15.0, 46.0],
    'cac':     [12.0, 10.0, 8.0,  0.0,  5.0,  2.0,  15.0],
})
ltv_demo['ltv90_vs_cac'] = ltv_demo['ltv_90'] - ltv_demo['cac']
ltv_demo['profitable'] = ltv_demo['ltv90_vs_cac'] > 0

print('\nLTV vs CAC by Source (Demonstration):')
print(ltv_demo.to_string(index=False))
print('\nUnprofitable channels (CAC > D90 LTV):')
print(ltv_demo[~ltv_demo['profitable']][['source','cac','ltv_90','ltv90_vs_cac']])

# Power-law LTV curve fit: ltv(t) = a * t^b
def power_ltv(t, a, b):
    return a * np.power(t, b)

x_days = np.array([30, 60, 90])
fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(ltv_demo)))
for i, row in ltv_demo.iterrows():
    y_vals = np.array([row['ltv_30'], row['ltv_60'], row['ltv_90']])
    try:
        popt, _ = curve_fit(power_ltv, x_days, y_vals, p0=[5.0, 0.5], maxfev=2000)
        x_fit   = np.linspace(1, 180, 200)
        ltv_180 = power_ltv(180, *popt)
        ax.plot(x_fit, power_ltv(x_fit, *popt), color=colors[i], alpha=0.8, label=f"{row['source']} (D180≈${ltv_180:.0f})")
        ax.scatter(x_days, y_vals, color=colors[i], s=60)
    except Exception:
        pass

ax.set_title('LTV Curve by Acquisition Source (Power-Law Fit)', fontsize=13, fontweight='bold')
ax.set_xlabel('Days Post-Install')
ax.set_ylabel('Cumulative LTV ($)')
ax.legend(loc='upper left', fontsize=8)
ax.axvline(90, color='gray', linestyle='--', alpha=0.5, label='D90 observation window')
plt.tight_layout()
plt.savefig('ltv_curves.png', dpi=120, bbox_inches='tight')
plt.show()


**Business Interpretation:**  
- **Referral** users have the highest LTV — they're vouched by existing high-value users (selection bias works in our favor here).  
- **TikTok** has lowest LTV despite reasonable install volume — younger demographic with lower purchase intent.  
- **Apple Search Ads** has highest CAC but D90 LTV justifies it (high-intent keyword targeting).  
- Power-law curve fit projects D180 LTV from 3 observations — useful for budget decisions before full data matures.

**Common Mistake:** Using average LTV without accounting for the survivor bias — users who make no purchases are zeros in LTV, not excluded.


---
### Q12 — Audience Overlap Analysis
**Difficulty:** Hard | **Time:** 15 min

**Business Context:**  
The paid team is running 5 targeting segments simultaneously. Overlapping audiences cause users to see multiple ads, increasing frequency fatigue and wasting budget through auction competition against ourselves.

**Problem Statement:**  
1. Compute pairwise Jaccard similarity between all 5 audience segments.  
2. Flag any pair with > 30% overlap.  
3. Estimate the "double-charged" users (users in both segments exposed to both campaigns simultaneously).  
4. Recommend which segments to consolidate.


In [ ]:
# ── Q12 Solution ───────────────────────────────────────────────────────────────

# segments dict is defined in Part 1:
# segments = {'high_ltv_top10': set(...), 'cart_abandoners': set(...), ...}

seg_names = list(segments.keys())
n_segs    = len(seg_names)

overlap_matrix = np.zeros((n_segs, n_segs))
jaccard_matrix = np.zeros((n_segs, n_segs))

for i in range(n_segs):
    for j in range(n_segs):
        set_i = segments[seg_names[i]]
        set_j = segments[seg_names[j]]
        intersection = len(set_i & set_j)
        union        = len(set_i | set_j)
        overlap_matrix[i, j] = intersection
        jaccard_matrix[i, j] = intersection / union if union > 0 else 0.0

overlap_df  = pd.DataFrame(overlap_matrix,  index=seg_names, columns=seg_names)
jaccard_df  = pd.DataFrame(jaccard_matrix.round(4), index=seg_names, columns=seg_names)

print('Pairwise User Overlap (count):')
print(overlap_df.astype(int).to_string())

print('\nPairwise Jaccard Similarity:')
print(jaccard_df.to_string())

# Flag high-overlap pairs
print('\nHigh-overlap pairs (Jaccard > 0.10):')
for i in range(n_segs):
    for j in range(i+1, n_segs):
        j_score = jaccard_matrix[i, j]
        if j_score > 0.10:
            shared  = int(overlap_matrix[i, j])
            print(f'  {seg_names[i]} <-> {seg_names[j]}: Jaccard={j_score:.3f}  shared_users={shared:,}')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(jaccard_df, annot=True, fmt='.3f', cmap='Oranges',
            linewidths=0.5, ax=axes[0], vmin=0, vmax=0.5)
axes[0].set_title('Audience Overlap — Jaccard Similarity', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

# Venn-like bar: segment sizes with overlap
sizes = [len(v) for v in segments.values()]
axes[1].barh(seg_names, sizes, color=sns.color_palette('husl', n_segs))
for i, (name, size) in enumerate(zip(seg_names, sizes)):
    axes[1].text(size + 20, i, f'{size:,}', va='center', fontsize=9)
axes[1].set_title('Segment Sizes', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Users')

plt.tight_layout()
plt.savefig('audience_overlap.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nRecommendation:')
print('  Consolidate high_ltv_top10 + cart_abandoners into a single "high_intent_buyers" segment')
print('  to reduce auction self-competition and frequency fatigue.')


---
### Q13 — Event Sequence Pattern Mining
**Difficulty:** Hard | **Time:** 20 min

**Business Context:**  
The product team wants to understand the most common 3-event sequences (trigrams) leading up to a purchase. This will inform feature prioritization and in-app nudges.

**Problem Statement:**  
1. For each user who made a purchase, extract the 3 events immediately preceding their first purchase.  
2. Count and rank the most frequent pre-purchase trigrams.  
3. Compare trigrams for high-LTV users (top 20%) vs low-LTV users.

**Constraint:** Only include the sequence from within the same session as the purchase.


In [ ]:
# ── Q13 Solution ────────────────────────────────────────────────────────────────

# Step 1: Identify purchasers and their purchase sessions
purchasers = set(purchases['user_id'])

evt_seq = app_events_clean[
    (app_events_clean['user_id'].isin(purchasers)) &
    (~app_events_clean['user_id'].isin(bots)) &
    (app_events_clean['event_timestamp'].notna())
][['user_id','session_id','event_name','event_timestamp']].copy()

evt_seq['event_timestamp'] = evt_seq['event_timestamp'].apply(
    lambda x: x.tz_localize('UTC') if (pd.notna(x) and getattr(x,'tzinfo',None) is None) else x
)
evt_seq = evt_seq.sort_values(['user_id','session_id','event_timestamp'])

# Step 2: Find the session containing first purchase
first_purchase = (
    purchases[purchases['user_id'].isin(purchasers)]
    .sort_values('purchase_time')
    .groupby('user_id')
    .first()
    .reset_index()[['user_id','purchase_time']]
)
first_purchase['purchase_time'] = pd.to_datetime(first_purchase['purchase_time'])
if first_purchase['purchase_time'].dt.tz is None:
    first_purchase['purchase_time'] = first_purchase['purchase_time'].dt.tz_localize('UTC')

# Join events with first purchase time — get events BEFORE purchase
evt_seq = evt_seq.merge(first_purchase, on='user_id', how='inner')
evt_seq = evt_seq[evt_seq['event_timestamp'] < evt_seq['purchase_time']]

# Step 3: Extract last 3 events per user (within 60 min of purchase)
evt_seq['min_to_purchase'] = (
    (evt_seq['purchase_time'] - evt_seq['event_timestamp'])
    .dt.total_seconds() / 60
)
evt_seq = evt_seq[evt_seq['min_to_purchase'] <= 60]

trigrams = []
for uid, grp in evt_seq.groupby('user_id'):
    evts = grp.sort_values('event_timestamp')['event_name'].tolist()
    if len(evts) >= 3:
        # last 3 events before purchase
        trigram = tuple(evts[-3:])
        trigrams.append({'user_id': uid, 'trigram': trigram})
    elif len(evts) >= 2:
        trigram = tuple(['(start)'] + evts[-2:])
        trigrams.append({'user_id': uid, 'trigram': trigram})

trigram_df = pd.DataFrame(trigrams)

# Step 4: Count and rank
top_trigrams = Counter(trigram_df['trigram']).most_common(15)
print('Top 15 Pre-Purchase Event Trigrams:')
for i, (trig, count) in enumerate(top_trigrams, 1):
    print(f'  {i:>2}. {" → ".join(trig):<60} count={count:,}')

# Step 5: LTV split
ltv_threshold = users['lifetime_value'].quantile(0.80)
high_ltv = set(users[users['lifetime_value'] >= ltv_threshold]['user_id'])
low_ltv  = set(users[users['lifetime_value'] <  ltv_threshold]['user_id'])

high_trig = Counter(trigram_df[trigram_df['user_id'].isin(high_ltv)]['trigram']).most_common(5)
low_trig  = Counter(trigram_df[trigram_df['user_id'].isin(low_ltv)]['trigram']).most_common(5)

print('\nTop 5 Trigrams — High-LTV Users (top 20%):')
for trig, count in high_trig:
    print(f'  {" → ".join(trig)}')

print('\nTop 5 Trigrams — Low-LTV Users:')
for trig, count in low_trig:
    print(f'  {" → ".join(trig)}')


**Business Interpretation:**  
- `video_watch → product_view → add_to_cart` as a top trigram validates the video-to-purchase hypothesis — invest in shoppable video content.  
- High-LTV users tend to use `search → product_view → purchase` (direct intent), while low-LTV users show more browsing patterns.  
- Use top trigrams to build in-app nudge triggers: detect `product_view → add_to_cart` and show a time-limited discount.

**Interviewer Note:** Ask candidates how they'd scale this to 1B events — answer should involve Spark N-gram analysis with window functions, not pandas groupby.


---
### Q14 — Ad Fatigue Analysis
**Difficulty:** Hard | **Time:** 18 min

**Business Context:**  
The media buyer suspects users are seeing the same ads too many times, causing CTR decline (ad fatigue). Analyze how CTR changes with increasing impression frequency.

**Problem Statement:**  
1. For each user × campaign combination, bucket users by how many times they've seen the campaign ad (frequency buckets: 1–2, 3–5, 6–10, 11–20, 20+).  
2. For each bucket, compute CTR.  
3. Identify at which frequency CTR starts declining significantly.  
4. Recommend a frequency cap.


In [ ]:
# ── Q14 Solution ────────────────────────────────────────────────────────────────

# User-campaign impression frequency
user_camp_impr = ad_impressions.groupby(['user_id','campaign_id']).agg(
    impressions = ('impression_id','count')
).reset_index()

# User-campaign click indicator
user_camp_click = ad_clicks.groupby(['user_id','campaign_id']).agg(
    clicks = ('click_id','count')
).reset_index()

freq_df = user_camp_impr.merge(user_camp_click, on=['user_id','campaign_id'], how='left')
freq_df['clicks'] = freq_df['clicks'].fillna(0)
freq_df['clicked'] = (freq_df['clicks'] > 0).astype(int)

# Frequency buckets
bins   = [0, 2, 5, 10, 20, 1000]
labels = ['1-2','3-5','6-10','11-20','21+']
freq_df['freq_bucket'] = pd.cut(freq_df['impressions'], bins=bins, labels=labels)

# CTR by bucket
fatigue_analysis = freq_df.groupby('freq_bucket', observed=True).agg(
    users         = ('user_id','nunique'),
    total_impr    = ('impressions','sum'),
    total_clicks  = ('clicks','sum')
).reset_index()
fatigue_analysis['CTR'] = (fatigue_analysis['total_clicks'] / fatigue_analysis['total_impr']).round(6)
fatigue_analysis['CTR_pct'] = (fatigue_analysis['CTR'] * 100).round(4)

print('Ad Fatigue Analysis — CTR by Impression Frequency:')
print(fatigue_analysis[['freq_bucket','users','total_impr','total_clicks','CTR_pct']].to_string(index=False))

# Peak CTR bucket
max_ctr_bucket = fatigue_analysis.loc[fatigue_analysis['CTR'].idxmax(), 'freq_bucket']
print(f'\nPeak CTR at frequency bucket: {max_ctr_bucket}')
print('Recommendation: Set frequency cap at the peak + 1 bucket to prevent fatigue.')

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(
    fatigue_analysis['freq_bucket'].astype(str),
    fatigue_analysis['CTR_pct'],
    color=[('#2ECC71' if b == str(max_ctr_bucket) else '#E74C3C') for b in fatigue_analysis['freq_bucket'].astype(str)]
)
for bar, val in zip(bars, fatigue_analysis['CTR_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.3f}%', ha='center', va='bottom', fontsize=10)
ax.set_title('CTR by Ad Impression Frequency (Ad Fatigue Curve)', fontsize=13, fontweight='bold')
ax.set_xlabel('Impression Frequency Bucket')
ax.set_ylabel('CTR (%)')
plt.tight_layout()
plt.savefig('ad_fatigue.png', dpi=120, bbox_inches='tight')
plt.show()


**Business Interpretation:**  
- Typical ad fatigue pattern: CTR peaks at 3–5 exposures, then declines sharply after 10+.  
- Setting a frequency cap at 6–8 per campaign per 7-day window is a standard best practice.  
- Recommendation: Implement automated frequency capping in campaign settings (Facebook/Google both support this) and rotate creative variants to delay fatigue onset.


---
### Q15 — Behavioral Cohort Analysis (KMeans Clustering)
**Difficulty:** Hard | **Time:** 20 min

**Business Context:**  
Instead of demographic segments, the product team wants behavioral cohorts based on in-app engagement patterns. Build 4 behavioral clusters and profile each one.

**Problem Statement:**  
1. Build per-user behavioral features: event counts, event diversity, session frequency, avg session length, purchase flag, subscription flag.  
2. Cluster users into 4 behavioral cohorts using KMeans.  
3. Name and characterize each cohort.  
4. Report cohort sizes, avg LTV, and conversion rates.


In [ ]:
# ── Q15 Solution ────────────────────────────────────────────────────────────────

# Feature 1: Event counts and diversity per user
evt_features = app_events_clean[
    (~app_events_clean['user_id'].isin(bots)) &
    (app_events_clean['event_timestamp'].notna())
].groupby('user_id').agg(
    total_events    = ('event_name','count'),
    unique_events   = ('event_name','nunique'),
    unique_sessions = ('session_id','nunique'),
    active_days     = ('event_timestamp', lambda x: x.apply(lambda ts: ts.date() if pd.notna(ts) else None).nunique())
).reset_index()

# Feature 2: Purchase and subscription flags
buyers     = set(purchases['user_id'])
subscribed = set(subscriptions['user_id'])
evt_features['has_purchased']   = evt_features['user_id'].isin(buyers).astype(int)
evt_features['has_subscribed']  = evt_features['user_id'].isin(subscribed).astype(int)

# Feature 3: Avg events per session
evt_features['events_per_session'] = (evt_features['total_events'] / evt_features['unique_sessions'].clip(lower=1)).round(2)

# Feature 4: LTV (from users table)
evt_features = evt_features.merge(users[['user_id','lifetime_value']], on='user_id', how='left')
evt_features['lifetime_value'] = evt_features['lifetime_value'].fillna(0)

# KMeans clustering
cluster_features = ['total_events','unique_events','unique_sessions','active_days',
                    'has_purchased','has_subscribed','events_per_session']
X_clust = evt_features[cluster_features].fillna(0).values
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X_clust)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
evt_features['cluster'] = kmeans.fit_predict(X_scaled)

# Profile clusters
cluster_profile = evt_features.groupby('cluster').agg(
    size              = ('user_id','count'),
    avg_events        = ('total_events','mean'),
    avg_sessions      = ('unique_sessions','mean'),
    avg_active_days   = ('active_days','mean'),
    pct_purchasers    = ('has_purchased','mean'),
    pct_subscribers   = ('has_subscribed','mean'),
    avg_ltv           = ('lifetime_value','mean')
).round(3)

cluster_profile['pct_purchasers'] = (cluster_profile['pct_purchasers'] * 100).round(1)
cluster_profile['pct_subscribers'] = (cluster_profile['pct_subscribers'] * 100).round(1)

print('Behavioral Cohort Profiles:')
print(cluster_profile.to_string())

# Auto-name clusters by engagement level
cluster_profile['engagement_score'] = (
    cluster_profile['avg_events'] / cluster_profile['avg_events'].max() * 0.4 +
    cluster_profile['pct_purchasers'] / 100 * 0.35 +
    cluster_profile['avg_active_days'] / cluster_profile['avg_active_days'].max() * 0.25
)
rank = cluster_profile['engagement_score'].rank(ascending=False).astype(int)
cluster_names = {1:'Power Users', 2:'Engaged Browsers', 3:'Casual Users', 4:'Churned/Dormant'}
cluster_profile['cohort_name'] = rank.map(cluster_names)

print('\nCohort Names:')
print(cluster_profile[['cohort_name','size','avg_ltv','pct_purchasers']].to_string())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cmap = ['#E74C3C','#3498DB','#2ECC71','#9B59B6']

axes[0].bar(cluster_profile.index.astype(str), cluster_profile['size'], color=cmap)
axes[0].set_title('Cluster Sizes', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Cluster ID')
axes[0].set_ylabel('Users')

axes[1].bar(cluster_profile.index.astype(str), cluster_profile['avg_ltv'], color=cmap)
axes[1].set_title('Average LTV by Cluster', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Cluster ID')
axes[1].set_ylabel('Avg LTV ($)')

plt.tight_layout()
plt.savefig('behavioral_clusters.png', dpi=120, bbox_inches='tight')
plt.show()


**Business Interpretation:**  
- **Power Users (Cluster 1):** High sessions, high LTV, subscribed — focus on retention and upsell. Do not expose to acquisition ads (wasted spend).  
- **Engaged Browsers (Cluster 2):** High browsing, low conversion — friction in checkout. A/B test checkout UX improvements.  
- **Casual Users (Cluster 3):** Low engagement — target with re-engagement push campaigns.  
- **Dormant (Cluster 4):** Very low activity — candidate for win-back email campaigns.


---
### Q16 — Simpson's Paradox in Campaign Conversion
**Difficulty:** Hard | **Time:** 15 min

**Business Context:**  
The CMO is presented with a report showing that Campaign A has a higher overall conversion rate than Campaign B. But when you break it down by platform (iOS vs Android), Campaign B outperforms Campaign A on BOTH platforms. Demonstrate and explain this paradox.

**Problem Statement:**  
1. Construct a synthetic but realistic scenario demonstrating Simpson's Paradox.  
2. Show the aggregated vs disaggregated conversion rates.  
3. Explain which number the CMO should trust and why.


In [ ]:
# ── Q16 Solution — Simpson's Paradox ──────────────────────────────────────────

# Construction: Campaign A targets iOS heavily (iOS converts better)
# Campaign B targets Android heavily (Android converts worse)
# → Campaign A's aggregate CVR is higher only because of platform composition, not campaign quality

simpson_data = pd.DataFrame([
    # campaign, platform, installs, purchases
    {'campaign': 'Campaign_A', 'platform': 'iOS',     'installs': 8000,  'purchases': 720},   # CVR = 9.0%
    {'campaign': 'Campaign_A', 'platform': 'Android', 'installs': 2000,  'purchases': 140},   # CVR = 7.0%
    {'campaign': 'Campaign_B', 'platform': 'iOS',     'installs': 2000,  'purchases': 190},   # CVR = 9.5%
    {'campaign': 'Campaign_B', 'platform': 'Android', 'installs': 8000,  'purchases': 600},   # CVR = 7.5%
])
simpson_data['CVR'] = (simpson_data['purchases'] / simpson_data['installs'] * 100).round(2)

print('Disaggregated View (by Platform):')
print(simpson_data.to_string(index=False))
print()
print('  → Campaign B beats Campaign A on BOTH iOS (9.5% > 9.0%) AND Android (7.5% > 7.0%)')

# Aggregated view
agg = simpson_data.groupby('campaign').agg(installs=('installs','sum'), purchases=('purchases','sum'))
agg['CVR_aggregate'] = (agg['purchases'] / agg['installs'] * 100).round(2)
print('\nAggregated View (ignoring platform):')
print(agg)
print()
print('  → Campaign A appears to win overall (8.6% > 7.9%) — PARADOX!')

print('\nExplanation:')
print('  Campaign A concentrates spend on iOS (8K of 10K installs) where CVR is inherently higher.')
print('  The apparent advantage is platform composition, NOT campaign quality.')
print('  Always stratify by confounders (platform, country, device_type) before comparing campaigns.')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Disaggregated
for i, (camp, grp) in enumerate(simpson_data.groupby('campaign')):
    x = [0 + i*0.35, 1 + i*0.35]
    axes[0].bar(x, grp.sort_values('platform')['CVR'], width=0.3,
                label=camp, color=['#3498DB','#E74C3C'][i])
axes[0].set_xticks([0.175, 1.175])
axes[0].set_xticklabels(['Android', 'iOS'])
axes[0].set_title('CVR by Platform (Disaggregated) — B wins both!', fontsize=11, fontweight='bold')
axes[0].set_ylabel('CVR (%)')
axes[0].legend()
axes[0].set_ylim(0, 13)

# Aggregated
agg.reset_index().plot.bar(x='campaign', y='CVR_aggregate', ax=axes[1],
                           color=['#3498DB','#E74C3C'], legend=False)
axes[1].set_title('CVR Aggregated — A appears to win! (WRONG)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('CVR (%)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle("Simpson's Paradox — Campaign Comparison", fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('simpsons_paradox.png', dpi=120, bbox_inches='tight')
plt.show()


**Business Interpretation:**  
- Always disaggregate by known confounders (platform, country, user segment) before declaring a winner.  
- The CMO should trust the **stratified** conversion rates, not the aggregate.  
- Real-world fix: normalize campaigns to the same platform/country mix before comparing (Cochran-Mantel-Haenszel test).

**Interviewer Note:** This is a classic trap question. Many senior analysts miss Simpson's Paradox in practice because they present aggregate numbers in dashboards.


---
### Q17 — A/B Test Statistical Significance for Campaign Comparison
**Difficulty:** Hard | **Time:** 15 min

**Business Context:**  
The growth team ran two versions of a retargeting campaign (A: standard creative, B: dynamic product ads). They claim B increased purchase conversion rate from 4.2% to 5.1%. Is this statistically significant?

**Problem Statement:**  
1. Run a two-proportion z-test.  
2. Compute 95% confidence intervals for each conversion rate.  
3. Calculate the minimum detectable effect (MDE) and required sample size.  
4. Check if the test was adequately powered.


In [ ]:
# ── Q17 Solution ────────────────────────────────────────────────────────────────
from scipy.stats import norm as sp_norm

# Observed data
n_a, conv_a = 4800, 202   # control: 4.2%
n_b, conv_b = 4700, 240   # treatment: 5.1%

p_a = conv_a / n_a
p_b = conv_b / n_b
print(f'Control (A):   n={n_a:,}  conversions={conv_a}  CVR={p_a*100:.2f}%')
print(f'Treatment (B): n={n_b:,}  conversions={conv_b}  CVR={p_b*100:.2f}%')
print(f'Observed lift: {(p_b - p_a)/p_a * 100:.1f}%  Absolute diff: {(p_b-p_a)*100:.2f}pp')

# Two-proportion z-test
p_pool  = (conv_a + conv_b) / (n_a + n_b)
se_pool = np.sqrt(p_pool * (1 - p_pool) * (1/n_a + 1/n_b))
z_stat  = (p_b - p_a) / se_pool
p_value = 1 - sp_norm.cdf(z_stat)   # one-tailed; use 2*(1-cdf) for two-tailed
p_two   = 2 * (1 - sp_norm.cdf(abs(z_stat)))

print(f'\nTwo-proportion Z-test:')
print(f'  z-statistic = {z_stat:.4f}')
print(f'  p-value (one-tailed) = {p_value:.6f}')
print(f'  p-value (two-tailed) = {p_two:.6f}')
if p_two < 0.05:
    print('  ✓ STATISTICALLY SIGNIFICANT at alpha=0.05')
else:
    print('  ✗ NOT statistically significant at alpha=0.05')

# 95% Confidence Intervals
def proportion_ci(p, n, z=1.96):
    margin = z * np.sqrt(p * (1-p) / n)
    return p - margin, p + margin

ci_a = proportion_ci(p_a, n_a)
ci_b = proportion_ci(p_b, n_b)
print(f'\n95% CIs:')
print(f'  A: [{ci_a[0]*100:.2f}%, {ci_a[1]*100:.2f}%]')
print(f'  B: [{ci_b[0]*100:.2f}%, {ci_b[1]*100:.2f}%]')

# MDE and sample size calculation
alpha, power = 0.05, 0.80
z_alpha = sp_norm.ppf(1 - alpha/2)
z_beta  = sp_norm.ppf(power)
mde     = 0.20  # 20% relative lift minimum
p1      = p_a
p2      = p_a * (1 + mde)
required_n = (
    (z_alpha * np.sqrt(2 * p1 * (1-p1)) + z_beta * np.sqrt(p1*(1-p1) + p2*(1-p2)))**2
    / (p2 - p1)**2
)
print(f'\nSample Size Calculation (MDE={mde*100:.0f}% relative lift):')
print(f'  Required n per arm: {int(required_n):,}')
print(f'  Current n A={n_a:,}  B={n_b:,}')
if min(n_a, n_b) >= required_n:
    print('  ✓ Test is adequately powered')
else:
    print(f'  ✗ Test is UNDERPOWERED (need {int(required_n):,} per arm, have {min(n_a,n_b):,})')

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
x = np.linspace(-4, 4, 1000)
ax.plot(x, sp_norm.pdf(x), 'b-', linewidth=2, label='Null distribution')
ax.axvline(z_stat, color='red', linewidth=2, linestyle='--', label=f'Observed z={z_stat:.2f}')
ax.axvline(z_alpha, color='green', linewidth=1.5, linestyle=':', label=f'Critical z={z_alpha:.2f} (α=0.05)')
ax.fill_between(x, sp_norm.pdf(x), where=(x > z_alpha), alpha=0.3, color='green', label='Rejection region')
ax.set_title('Z-Test: Campaign A vs B Conversion Rate', fontsize=12, fontweight='bold')
ax.set_xlabel('Z-score')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.savefig('ab_test.png', dpi=120, bbox_inches='tight')
plt.show()


**Business Interpretation:**  
- If significant: Dynamic Product Ads outperform standard creative by ~21% relative lift — roll out to 100% of retargeting budget.  
- If not significant: The observed difference may be noise. Run for more time or increase sample size.  
- **Danger:** Never call a winner mid-test (peeking problem inflates false positive rate to ~20% at 5 peeks).

**Common Mistakes:**  
- Using a t-test for proportions (should use z-test or chi-squared).  
- Reporting p-value without confidence intervals (p-value alone doesn't tell you the magnitude).  
- Not pre-registering sample size — post-hoc power analysis is misleading.


---
### Q18 — Reactivation & Win-Back Analysis
**Difficulty:** Hard | **Time:** 15 min

**Business Context:**  
The CRM team is running a win-back email campaign (CMP_011) targeting users who haven't opened the app in 30+ days. Evaluate its effectiveness.

**Problem Statement:**  
1. Define churned users: no `app_open` event for 30+ days before the campaign start date.  
2. Compute win-back rate = % of churned users who re-opened the app within 7 days of the email campaign start.  
3. Compare win-back rate by country and acquisition source.  
4. Estimate revenue impact of win-back.


In [ ]:
# ── Q18 Solution ────────────────────────────────────────────────────────────────

CAMPAIGN_START = pd.Timestamp('2024-02-15', tz='UTC')
CHURN_WINDOW   = 30  # days inactive = churned
WINBACK_WINDOW = 7   # days to measure re-activation

# Last app_open per user (before campaign start)
app_opens_all = app_events_clean[
    (app_events_clean['event_name'] == 'app_open') &
    (app_events_clean['event_timestamp'].notna())
][['user_id','event_timestamp']].copy()
app_opens_all['event_timestamp'] = app_opens_all['event_timestamp'].apply(
    lambda x: x.tz_localize('UTC') if (pd.notna(x) and getattr(x,'tzinfo',None) is None) else x
)

last_open = app_opens_all[app_opens_all['event_timestamp'] < CAMPAIGN_START].groupby('user_id')['event_timestamp'].max().reset_index()
last_open.columns = ['user_id','last_open_before_campaign']

last_open['days_inactive'] = (
    (CAMPAIGN_START - last_open['last_open_before_campaign']).dt.days
)

# Churned users = inactive 30+ days
churned = last_open[last_open['days_inactive'] >= CHURN_WINDOW].merge(
    users[['user_id','country','acquisition_source','is_bot']], on='user_id', how='left'
)
churned = churned[~churned['is_bot']]
print(f'Churned users (30+ days inactive): {len(churned):,}')

# Re-activation: app_open within 7 days AFTER campaign start
post_campaign = app_opens_all[
    (app_opens_all['event_timestamp'] >= CAMPAIGN_START) &
    (app_opens_all['event_timestamp'] <= CAMPAIGN_START + pd.Timedelta(days=WINBACK_WINDOW))
]
reactivated = set(post_campaign['user_id'])

churned['reactivated'] = churned['user_id'].isin(reactivated).astype(int)
overall_rate = churned['reactivated'].mean()
print(f'Overall win-back rate: {overall_rate*100:.2f}%')

# By country
country_wb = churned.groupby('country').agg(
    churned_users  = ('user_id','count'),
    reactivated    = ('reactivated','sum')
).reset_index()
country_wb['winback_rate'] = (country_wb['reactivated'] / country_wb['churned_users'] * 100).round(2)
country_wb = country_wb.sort_values('winback_rate', ascending=False)
print('\nWin-back rate by country:')
print(country_wb.head(8).to_string(index=False))

# Revenue impact
post_rev = purchases[
    pd.to_datetime(purchases['purchase_time']).dt.tz_localize('UTC', ambiguous='NaT', nonexistent='NaT')
    .between(CAMPAIGN_START, CAMPAIGN_START + pd.Timedelta(days=WINBACK_WINDOW))
]
post_rev_reactivated = post_rev[post_rev['user_id'].isin(reactivated)]
total_winback_rev = post_rev_reactivated['revenue_usd'].sum()
print(f'\nRevenue from reactivated users in 7-day window: ${total_winback_rev:,.2f}')
print(f'Average revenue per reactivated user: ${total_winback_rev / max(len(reactivated & set(churned["user_id"])), 1):.2f}')


---
### Q19 — Engagement Score Construction
**Difficulty:** Medium-Hard | **Time:** 15 min

**Business Context:**  
The product team needs a composite engagement score (0–100) for each user to power real-time personalization and push notification targeting.

**Problem Statement:**  
1. Define weighted engagement score from:  
   - Recency (days since last app_open): 30% weight  
   - Frequency (sessions in last 30 days): 25% weight  
   - Depth (unique event types): 20% weight  
   - Monetization (has purchased): 15% weight  
   - Virality (invite_sent count): 10% weight  
2. Normalize each component to 0–1, compute weighted sum, scale to 0–100.  
3. Segment users into: Cold (0–30), Warm (30–60), Hot (60–80), Champion (80–100).


In [ ]:
# ── Q19 Solution ────────────────────────────────────────────────────────────────
AS_OF_DATE = pd.Timestamp('2024-06-30', tz='UTC')

# Recency: days since last app_open
recency = app_opens_all.groupby('user_id')['event_timestamp'].max().reset_index()
recency.columns = ['user_id','last_open']
recency['days_since_last_open'] = (AS_OF_DATE - recency['last_open']).dt.days.clip(lower=0)

# Frequency: sessions in last 30 days
last_30_start = AS_OF_DATE - pd.Timedelta(days=30)
freq_events = app_events_clean[
    app_events_clean['event_timestamp'].apply(
        lambda x: (pd.notna(x) and
                   (x.tz_localize('UTC') if getattr(x,'tzinfo',None) is None else x) >= last_30_start)
    )
].groupby('user_id')['session_id'].nunique().reset_index()
freq_events.columns = ['user_id','sessions_30d']

# Depth: unique event types (all time)
depth = app_events_clean.groupby('user_id')['event_name'].nunique().reset_index()
depth.columns = ['user_id','unique_event_types']

# Monetization
monetization = purchases.groupby('user_id')['purchase_id'].count().reset_index()
monetization.columns = ['user_id','purchase_count']

# Virality: invite_sent events
virality = app_events_clean[
    app_events_clean['event_name'] == 'invite_sent'
].groupby('user_id').size().reset_index()
virality.columns = ['user_id','invite_count']

# Combine
eng = users[~users['is_bot']][['user_id']].copy()
eng = eng.merge(recency[['user_id','days_since_last_open']], on='user_id', how='left')
eng = eng.merge(freq_events, on='user_id', how='left')
eng = eng.merge(depth,       on='user_id', how='left')
eng = eng.merge(monetization, on='user_id', how='left')
eng = eng.merge(virality,    on='user_id', how='left')
eng = eng.fillna({'days_since_last_open': 999, 'sessions_30d': 0,
                  'unique_event_types': 0, 'purchase_count': 0, 'invite_count': 0})

# Normalize each component 0–1
def minmax(s): return (s - s.min()) / (s.max() - s.min() + 1e-9)

eng['recency_score']  = 1 - minmax(eng['days_since_last_open'])   # lower days = better
eng['freq_score']     = minmax(eng['sessions_30d'])
eng['depth_score']    = minmax(eng['unique_event_types'])
eng['monet_score']    = minmax(eng['purchase_count'])
eng['viral_score']    = minmax(eng['invite_count'])

# Weighted composite
eng['engagement_score'] = (
    0.30 * eng['recency_score'] +
    0.25 * eng['freq_score']    +
    0.20 * eng['depth_score']   +
    0.15 * eng['monet_score']   +
    0.10 * eng['viral_score']
) * 100

eng['segment'] = pd.cut(
    eng['engagement_score'],
    bins=[0, 30, 60, 80, 100],
    labels=['Cold','Warm','Hot','Champion'],
    include_lowest=True
)

print('Engagement Score Distribution:')
print(eng['segment'].value_counts().sort_index())
print(f'\nMean engagement score: {eng["engagement_score"].mean():.1f}')
print(f'Median: {eng["engagement_score"].median():.1f}')

# Viz
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
eng['engagement_score'].hist(bins=40, ax=axes[0], color='#3498DB', edgecolor='white')
axes[0].set_title('Engagement Score Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Score (0–100)')

seg_counts = eng['segment'].value_counts().sort_index()
seg_counts.plot.bar(ax=axes[1], color=['#E74C3C','#F39C12','#2ECC71','#9B59B6'])
axes[1].set_title('Users by Engagement Segment', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Segment')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('engagement_segments.png', dpi=120, bbox_inches='tight')
plt.show()


---
### Q20 — Incrementality Test (Holdout Group Analysis)
**Difficulty:** Expert | **Time:** 20 min

**Business Context:**  
The CMO wants to know the true incremental lift from the Facebook retargeting campaign (CMP_003). Without a holdout group, last-click attribution over-credits retargeting (it just captures users who would have converted organically anyway).

**Problem Statement:**  
1. Simulate a holdout experiment: randomly assign 20% of the retargeting-eligible audience to a holdout group (shown house ads, not campaign ads).  
2. Measure conversion rate in exposed vs holdout groups.  
3. Calculate true incremental CVR lift.  
4. Estimate incremental revenue generated by the campaign.
5. Compare to what last-click attribution would have reported.


In [ ]:
# ── Q20 Solution — Incrementality / Holdout Analysis ──────────────────────────

# Retargeting eligible = cart_abandoners segment
retarget_eligible = np.array(list(segments['cart_abandoners']))

# Assign 20% to holdout randomly
holdout_mask = rng.random(len(retarget_eligible)) < 0.20
holdout_ids  = set(retarget_eligible[holdout_mask])
exposed_ids  = set(retarget_eligible[~holdout_mask])

print(f'Exposed group:  {len(exposed_ids):,} users')
print(f'Holdout group:  {len(holdout_ids):,} users')

# Simulate different conversion rates:
# Exposed group: exposed to CMP_003 → higher conversion (but partially driven by intent)
# Holdout:       NOT exposed → baseline intent-driven conversion

# In a real experiment you'd observe these from event data.
# Here we simulate with realistic assumptions:
#   Baseline CVR (holdout):  6.5%  (high-intent users would convert anyway)
#   Exposed CVR:             8.2%  (some uplift from the ad)
#   Last-click CVR claimed: 11.0%  (all conversions credited to the ad)

np.random.seed(2024)

n_exposed = len(exposed_ids)
n_holdout = len(holdout_ids)

# Simulate conversions
exposed_conversions = int(n_exposed * 0.082)
holdout_conversions = int(n_holdout * 0.065)

exposed_cvr  = exposed_conversions / n_exposed
holdout_cvr  = holdout_conversions / n_holdout
incremental_cvr = exposed_cvr - holdout_cvr
incrementality_pct = incremental_cvr / holdout_cvr * 100

last_click_cvr_claimed = 0.110   # from attribution report
last_click_inflation   = (last_click_cvr_claimed - exposed_cvr) / exposed_cvr * 100

print(f'\nIncrementality Test Results:')
print(f'  Holdout CVR  (baseline intent): {holdout_cvr*100:.2f}%')
print(f'  Exposed CVR  (with campaign):   {exposed_cvr*100:.2f}%')
print(f'  Incremental CVR lift:           {incremental_cvr*100:.2f}pp  ({incrementality_pct:.1f}% relative)')
print(f'  Last-click claimed CVR:         {last_click_cvr_claimed*100:.1f}% ← INFLATED')
print(f'  Last-click inflation:           {last_click_inflation:.1f}%')

# Revenue impact
avg_order_value = purchases['revenue_usd'].median()
true_incremental_users  = incremental_cvr * n_exposed
true_incremental_rev    = true_incremental_users * avg_order_value
last_click_claimed_rev  = last_click_cvr_claimed * n_exposed * avg_order_value

print(f'\nRevenue Attribution Comparison:')
print(f'  True incremental users: {true_incremental_users:.0f}')
print(f'  True incremental revenue: ${true_incremental_rev:,.2f}')
print(f'  Last-click attributed revenue: ${last_click_claimed_rev:,.2f}')
print(f'  Attribution inflation: {(last_click_claimed_rev - true_incremental_rev)/true_incremental_rev*100:.1f}%')

# Statistical significance
from scipy.stats import chi2_contingency
cont = np.array([
    [exposed_conversions,  n_exposed - exposed_conversions],
    [holdout_conversions,  n_holdout - holdout_conversions]
])
chi2, p_val, _, _ = chi2_contingency(cont)
print(f'\nChi-squared test: chi2={chi2:.3f}  p={p_val:.6f}')
if p_val < 0.05:
    print('  ✓ Incremental lift is statistically significant')
else:
    print('  ✗ Cannot confirm significant lift — increase test duration')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CVR comparison
bars = axes[0].bar(
    ['Holdout\n(Baseline)','Exposed\n(True CVR)','Last-Click\n(Claimed)'],
    [holdout_cvr*100, exposed_cvr*100, last_click_cvr_claimed*100],
    color=['#95A5A6','#2ECC71','#E74C3C']
)
for bar, val in zip(bars, [holdout_cvr*100, exposed_cvr*100, last_click_cvr_claimed*100]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('CVR: True Incrementality vs Last-Click', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Conversion Rate (%)')

# Revenue attribution
axes[1].bar(
    ['True Incremental\nRevenue','Last-Click\nAttributed Revenue'],
    [true_incremental_rev, last_click_claimed_rev],
    color=['#2ECC71','#E74C3C']
)
axes[1].set_title('Revenue: True vs Last-Click Attribution', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Revenue ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('incrementality.png', dpi=120, bbox_inches='tight')
plt.show()


**Business Interpretation:**  
- True incremental lift (~1.7pp CVR) is real but significantly lower than last-click implies (~4.5pp).  
- Last-click attribution inflates retargeting ROI by ~34% — the campaign was profitable but far less so than the dashboard showed.  
- Recommendation: Run geo-based holdouts (cheaper than user-level) or use Facebook's Conversion Lift feature for ongoing incrementality measurement.

**Interviewer Note:** This is the hardest question. Strong candidates will mention *SUTVA violations* (stable unit treatment value assumption) — users in the exposed group might tell holdout users about deals, contaminating the experiment.


---
## Section 6 — Debugging Round

Each snippet below contains 1–3 intentional bugs. Identify, explain, and fix them.


### Bug 1 — Retention Calculation (Wrong Denominator + Incorrect Join)
**Symptoms:** D7 retention appears higher than D1 for some cohorts.


In [ ]:
# ── BROKEN CODE — Identify the bugs ───────────────────────────────────────────

# Bug version:
def broken_retention(installs_df, events_df, day):
    installs_df['cohort'] = pd.to_datetime(installs_df['install_time']).dt.to_period('M')

    # BUG 1: Should be left join, not inner join — drops users who never returned
    merged = installs_df.merge(events_df, on='user_id', how='inner')

    merged['days'] = (pd.to_datetime(merged['event_timestamp']) - pd.to_datetime(merged['install_time'])).dt.days

    # BUG 2: Denominator uses retained users, not all cohort users — inflates %
    retained_per_cohort = merged[merged['days'] == day].groupby('cohort')['user_id'].nunique()
    denominator = merged[merged['days'] == day].groupby('cohort')['user_id'].nunique()  # WRONG!

    return retained_per_cohort / denominator

print('Bug 1 — broken_retention() has 2 bugs:')
print('  BUG 1: inner join drops users who never returned → denominator too small → inflated %')
print('  BUG 2: denominator is same as numerator (retained) instead of total cohort size → always 100%')


In [ ]:
# ── FIXED CODE ────────────────────────────────────────────────────────────────
def fixed_retention(installs_df, events_df, day, tolerance=1):
    inst = installs_df.copy()
    inst['cohort'] = pd.to_datetime(inst['install_time']).dt.to_period('M')

    # FIX 1: Use left join to preserve all installed users
    merged = inst.merge(events_df[['user_id','event_timestamp','event_name']], on='user_id', how='left')
    merged = merged[merged['event_name'] == 'app_open']   # only retention-qualifying events

    # Need UTC-aware timestamps
    merged['event_timestamp'] = pd.to_datetime(merged['event_timestamp'])
    merged['install_time']    = pd.to_datetime(merged['install_time'])

    merged['days'] = (merged['event_timestamp'] - merged['install_time']).dt.days

    # FIX 2: Correct denominator = all installed users in cohort
    cohort_sizes = inst.groupby('cohort')['user_id'].nunique().rename('cohort_size')
    retained     = merged[merged['days'].between(day - tolerance, day + tolerance)].groupby('cohort')['user_id'].nunique().rename('retained')

    result = pd.DataFrame({'cohort_size': cohort_sizes}).join(retained).fillna(0)
    result['retention_rate'] = (result['retained'] / result['cohort_size']).round(4)
    return result

print('Fixed retention function — uses left join + correct denominator (total cohort size).')
print('Key change: denominator = cohort_size (all installs), not retained count.')


### Bug 2 — Merge Explosion (Cartesian Product)
**Symptom:** DataFrame after merge has 100× more rows than expected.


In [ ]:
# ── BROKEN CODE ───────────────────────────────────────────────────────────────
print('Bug 2 — Merge Explosion:')
print('''
# Broken:
purchases_with_events = purchases.merge(
    app_events_clean,
    on='user_id',       # BUG: joining on user_id only — many events per user
    how='left'           # Result: N_purchases × N_events_per_user rows = EXPLOSION
)
''')
print('Diagnose: print df.shape before and after merge.')
print('Check: if result rows >> left table rows, you have a one-to-many explosion.\n')

print('FIX OPTIONS:')
print('  1. Aggregate events FIRST, then merge:')
print('''
    event_summary = app_events_clean.groupby('user_id').agg(
        total_events=('event_name','count'),
        last_event=('event_timestamp','max')
    ).reset_index()
    purchases_enriched = purchases.merge(event_summary, on='user_id', how='left')
''')
print('  2. Use merge_asof for time-based joins (events closest to purchase time).')
print('  3. Always validate: assert len(result) == len(left_table), f"merge explosion: {len(result)}"')


### Bug 3 — Attribution Window Off-By-One (Timezone Bug)
**Symptom:** Attribution rate drops 30% unexpectedly when code is run on different machines.


In [ ]:
# ── BROKEN CODE ───────────────────────────────────────────────────────────────
print('Bug 3 — Timezone Bug in Attribution Window:')
print('''
# Broken:
install_ts    = pd.to_datetime(installs["install_time"])         # timezone-naive
click_ts      = pd.to_datetime(ad_clicks["click_time"])           # UTC-aware
days_diff     = (install_ts - click_ts).dt.days                   # TypeError or wrong result
valid_window  = days_diff.between(0, 7)                           # silently wrong on some systems
''')

print('ROOT CAUSE: Subtracting timezone-naive from timezone-aware Timestamps raises')
print('  TypeError in newer pandas. In older versions, it silently produces wrong values.')
print('  This explains the "drops 30% on different machines" — different pandas versions!')

print('\nFIX:')
print('''
# Correct:
def to_utc(col):
    ts = pd.to_datetime(col)
    if ts.dt.tz is None:
        return ts.dt.tz_localize("UTC")
    return ts.dt.tz_convert("UTC")

install_utc = to_utc(installs["install_time"])
click_utc   = to_utc(ad_clicks["click_time"])

# Now subtraction is safe
days_diff   = (install_utc - click_utc).dt.total_seconds() / 86400
valid_window = days_diff.between(0, 7)
''')
print('RULE: Always normalize ALL timestamps to UTC before any time arithmetic.')


### Bug 4 — SettingWithCopyWarning + Silent Data Mutation


In [ ]:
# ── BROKEN CODE ───────────────────────────────────────────────────────────────
print('Bug 4 — SettingWithCopyWarning:')
print('''
# Broken:
paid_users = users[users["acquisition_source"] != "organic"]
paid_users["is_paid"] = True     # SettingWithCopyWarning!
                                  # Might not modify the original DataFrame
                                  # Behavior is undefined (implementation-dependent)
''')

print('ROOT CAUSE: pandas boolean indexing returns a COPY or VIEW depending on the operation.')
print('  Assigning to a copy does not modify the original and may produce incorrect results.')
print('  The warning exists because behavior is undefined.')

print('\nFIX 1 — Use .copy() explicitly:')
print('''
    paid_users = users[users["acquisition_source"] != "organic"].copy()
    paid_users["is_paid"] = True    # Safe — modifying an explicit copy
''')

print('FIX 2 — Use .loc on the original DataFrame:')
print('''
    users.loc[users["acquisition_source"] != "organic", "is_paid"] = True
''')

print('FIX 3 (pandas 2.0+) — Copy-on-Write is the default; behavior is now predictable.')
print('  But .copy() is still best practice for clarity.')


### Bug 5 — Incorrect Funnel Calculation (Order Not Enforced)


In [ ]:
# ── BROKEN CODE ───────────────────────────────────────────────────────────────
print('Bug 5 — Funnel Calculation Without Enforcing Order:')
print('''
# Broken approach — counts users who EVER did each event, ignoring order:
FUNNEL = ["product_view", "add_to_cart", "purchase"]
for step in FUNNEL:
    count = app_events_clean[app_events_clean["event_name"] == step]["user_id"].nunique()
    print(f"{step}: {count:,} users")
# BUG: A user who did purchase BEFORE product_view is counted as a converter.
# This massively inflates conversion rates (typical inflation: 20-40%).
''')

print('ROOT CAUSE: Funnel analysis requires events to occur IN ORDER.')
print('  Simply counting users who ever fired each event ignores temporal sequence.')

print('\nFIX — Enforce temporal ordering:')
print('''
    # For each user, find the FIRST time they did each step.
    # Step N timestamp must be AFTER step N-1 timestamp.
    
    step_times = {}
    for step in FUNNEL:
        step_times[step] = (
            app_events_clean[app_events_clean["event_name"] == step]
            .groupby("user_id")["event_timestamp"].min()
        )
    
    # Build ordered funnel
    funnel_df = pd.DataFrame(step_times)
    # Each step must occur after previous step
    valid_pv  = funnel_df["product_view"].notna()
    valid_atc = valid_pv & (funnel_df["add_to_cart"] > funnel_df["product_view"])
    valid_pur = valid_atc & (funnel_df["purchase"]   > funnel_df["add_to_cart"])
    
    print(f"product_view: {valid_pv.sum():,}")
    print(f"add_to_cart:  {valid_atc.sum():,}")
    print(f"purchase:     {valid_pur.sum():,}")
''')
print('KEY LESSON: Always sort by timestamp and enforce step ordering in funnel analysis.')


---
## Section 7 — Final Business Summary & Recommendations

### Key Findings

| Area | Finding | Impact |
|------|---------|--------|
| **Fraud** | True bot rate ~8% vs 5% AppsFlyer-flagged | $32K/month wasted spend |
| **Attribution** | Last-click inflates retargeting ROAS by ~34% | Budget misallocation across channels |
| **Funnel** | Biggest drop: `add_to_cart → checkout_start` (~45% drop) | Revenue opportunity #1 |
| **Retention** | D7 retention below 15% (benchmark: 20%) | LTV ceiling hit early |
| **Ad Fatigue** | CTR declines after 5+ exposures | Frequency cap missing |
| **Segmentation** | 3 behavioral clusters driving 80% of revenue | Most spend targeting wrong clusters |
| **Attribution Paradox** | Simpson's Paradox present in platform-level CVR | Wrong campaign winner declared |


In [ ]:
# ── Final Business Recommendations Dashboard ──────────────────────────────────

recommendations = {
    'Campaign': [
        'Set frequency cap at 6 impressions per user per 7 days across all campaigns',
        'Kill YouTube Brand (CMP_014) and Influencer (CMP_006) on last-click — re-evaluate with holdout',
        'Increase Apple Search Ads budget by 25% — highest quality score and D90 LTV',
        'Consolidate cart_abandoners + high_ltv segments to reduce self-auction competition',
        'Implement geo-holdouts for all retargeting campaigns to measure true incrementality',
    ],
    'Product': [
        'A/B test checkout UX — 45% drop at checkout_start is the biggest revenue leak',
        'Launch video-to-purchase flow: top pre-purchase sequence is video_watch → product_view → purchase',
        'Implement in-app nudge after 3rd product_view with no add_to_cart (10% discount trigger)',
        'Build behavioral cohort-based push notification targeting (replace demographic targeting)',
        'Improve D1 onboarding — every 5pp D1 improvement yields ~$0.8 incremental D90 LTV',
    ],
    'Attribution & Tracking': [
        'Replace last-click with linear attribution as an interim step',
        'Implement Shapley value attribution for top 5 campaigns (ML-based)',
        'Fix timezone normalization in event pipeline — standardize all events to UTC at ingestion',
        'Add server-side event deduplication with 30-second window',
        'Deploy real-time bot detection using IsolationForest on event velocity features',
    ],
    'Growth Strategy': [
        'Referral program (CMP_013) has highest LTV — 3× budget increase recommended',
        'TikTok (CMP_009) shows low LTV — shift budget to higher-LTV channels unless D180 LTV improves',
        'Email reactivation ROI is positive — scale list to 3× with look-alike modeling',
        'Win-back campaign effective in US/UK — expand to CA and AU with localized creatives',
    ]
}

for category, items in recommendations.items():
    print(f'\n{category.upper()} RECOMMENDATIONS')
    print('=' * 60)
    for i, item in enumerate(items, 1):
        print(f'  {i}. {item}')

print('\n' + '='*60)
print('TRACKING IMPROVEMENTS')
print('='*60)
tracking_issues = [
    'Standardize event names at SDK level (camelCase → snake_case)',
    'Add mandatory fields: session_id, timezone, app_version to every event',
    'Implement server-side deduplication before writing to data warehouse',
    'Fix missing timestamps by injecting server-timestamp as fallback',
    'Add attribution delay buffer (48h) to account for offline purchases',
]
for i, item in enumerate(tracking_issues, 1):
    print(f'  {i}. {item}')


---
## Interviewer Evaluation Rubric

| Dimension | Weight | What to Look For |
|-----------|--------|------------------|
| **Analytical Thinking** | 25% | Asks clarifying questions, challenges assumptions, identifies confounders |
| **pandas Expertise** | 20% | Correct merge types, vectorized operations, avoids SettingWithCopyWarning |
| **Product Intuition** | 20% | Business interpretation of numbers, not just technical correctness |
| **Statistical Reasoning** | 15% | Correct significance tests, mentions effect size and power, not just p-value |
| **Debugging Ability** | 10% | Identifies root cause, not just symptom |
| **Production Thinking** | 10% | Mentions scale, memory, latency; proposes Spark/streaming alternatives |

### Red Flags
- Drops null rows without investigating why they're null
- Uses inner join when left join is appropriate (silent data loss)
- Claims campaign winner without statistical testing
- Doesn't ask about attribution model before computing ROAS
- Can't explain why Simpson's Paradox occurs

### Green Flags
- Asks "what's the business decision we're trying to make?" before coding
- Proposes holdout groups when asked about campaign effectiveness
- Handles timezone inconsistencies proactively
- Estimates memory requirements for large joins
- Distinguishes correlation from causation in event analysis

---
*End of Interview Assessment — ShopFlash Analytics*
